<div dir="rtl">

# رافِدَين ٤.٠ — التدريب الآلي (Autopilot) على RunPod RTX 5090

**رحلة كاملة** (تنزيل بيانات عراقية ← تجهيز ← تدريب ← تقييم ← حفظ دائم) تشغّلها **شبكة تجارب يديرها مشغّل آلي**: توزيع تلقائي على البطاقات، نقاط تفتيش مُتحقق من سلامتها، **إيقاف مبكر** عند استواء الخسارة، **حارس NaN** يقتل ويعيد الإطلاق، **سقف ميزانية** يحمي محفظتك، ثم **اختيار البطل تلقائياً بالأرقام**.

**بطاقة واحدة أو بطاقتان — نفس الدفتر:** ببطاقة 5090 واحدة يعمل بالوضع التسلسلي الآلي (تجربة تلو الأخرى بلا تدخل)، وببطاقتين يتحول تلقائياً للوضع الموازي (تجربة على كل بطاقة).

---

## مواصفات الحاوية المستهدفة

| المواصفة | القيمة | كيف يستغلها الدفتر؟ |
|---|---|---|
| GPU | **RTX 5090** (بطاقة واحدة أساساً — أو 2 اختيارياً) | المشغّل الآلي يوزع التجارب: تسلسلي على بطاقة، موازٍ على بطاقتين |
| ذاكرة GPU | **32GB VRAM** | دفعة **6** لكل تجربة |
| المعالج | **16 vCPU** | 4 عمال تحميل لكل تجربة جارية |
| الذاكرة | **92GB RAM** | تجربتان متزامنتان (في وضع البطاقتين) بلا أي ضغط |
| السعر | **$1.19/ساعة للبطاقة** | عدّاد كلفة مباشر يضربها بعدد بطاقاتك الفعلي |

> **القالب الذي رأيته في لوحة RunPod صحيح تماماً:** «RunPod PyTorch 2.8.0 · CUDA 12.8.1 · Ubuntu 24.04 · نسخة القالب 2.1.0-2.8.0» — PyTorch 2.8.0 مع CUDA 12.8 هو بالضبط ما تحتاجه معمارية Blackwell لبطاقة 5090، وهذا الدفتر مبني عليه ومختبر معه.

> **سألت عن قائمة «Available CUDA versions: 13.2 فقط»؟** لا تقلق ولا تتردد — اخترها وامضِ. السبب: هذا الدفتر يبني **بيئة Python معزولة خاصة به** (الخلية 4) تثبّت PyTorch 2.8.0 cu128 **بملفاته الثنائية الكاملة** (مكتبات CUDA 12.8 مدمجة داخل حزم pip نفسها)، فلا تعتمد على نسخة CUDA المثبتة في الحاوية إطلاقاً. المهم الوحيد هو مشغّل NVIDIA (الدرايفر) وهو أحدث من كليهما. كما أننا لا نترجم أي نواة CUDA بأنفسنا. النتيجة: اختيار 13.2 (أو أي شيء آخر) لا يمس التدريب بشيء.

---

## ما الجديد في الإصدار 4.0 (Autopilot)

| # | الميزة | ماذا تعني لك؟ |
|---|---|---|
| 1 | **نقاط تفتيش محصّنة** | حفظ كل 250 خطوة (رسمي) + فحص سلامة كل نقطة + حذف التالفة تلقائياً — لا استئناف من ملف مكسور |
| 2 | **الإيقاف المبكر** | تجربة استوت خسارتها تُوقف ويُصنع نموذجها — توفير وقت ومال حقيقي |
| 3 | **تصنيع النماذج** | أي توقف غير طبيعي ينتج `ft_model.pth` صالحاً بصيغة المدرب الرسمية من آخر نقطة سليمة + توثيق كامل في `model_manifest.json` |
| 4 | **حارس NaN نشط** | المدرب الرسمي لا يموت عند NaN بل يدرّب فاسداً — المشغّل يقتلها فوراً ويستأنف من نقطة سليمة |
| 5 | **الجرعة الأمينة** | عند الاستئناف تُطلق التجربة بالخطوات المتبقية فقط (لا جرعة مضاعفة) — مقارنة الوصفات تبقى عادلة |
| 6 | **سقف الميزانية** | $15 أو 6 ساعات فعلية (قابلة للتعديل) — صمام أمان مالي لا يسمح بمفاجآت الفاتورة |
| 7 | **نبض المشغّل** | لوحة المراقبة تفحص نبض المشغّل وتنبهك فوراً إن تعطل مع خطوة الإصلاح |
| 8 | **وضع البطاقة الواحدة** | مدعوم رسمياً كوضع أول — لا حاجة لبطاقتين للحصول على كل الميزات |

---

## كيف يعمل هذا الدفتر؟

| الخطوة | الخلية | ماذا يحدث؟ |
|---|---|---|
| 1. شبكة التجارب | 14 | تُعرَّف 4 تجارب (جرعة/دفعة/كمية بيانات + خضوع للإيقاف المبكر) |
| 2. المشغّل الآلي | 15 | عملية خلفية مستقلة: توزيع + نقاط تفتيش + إيقاف مبكر + حارس NaN + ميزانية |
| 3. المراقبة | 16 | لوحة حية: تقدم فعلي، حالة الإيقاف المبكر، الميزانية، نبض المشغّل، استغلال البطاقات |
| 4. اختيار البطل | 20 | بعد تقييم A/B موضوعي: صاحب أعلى تحسن بالتشابه بشرط سلامة المحتوى — ويمكنك فرض اختيارك |

**الأبعاد المتاحة للتجارب** (عبر واجهة `train.py` الرسمية فقط): عدد الخطوات، حجم الدفعة، كمية البيانات. معدل التعلم مثبّت **داخل** المدرب الرسمي (1e-5) ولا نلمسه — هذا جزء من الضمان.

---

## ضمانات صادقة — اقرأها مرة واحدة

**ما يضمنه هذا الدفتر حرفياً:**
- كل تجربة تستدعي `train.py` **الرسمي** دون أي تعديل، من نفس نقطة الأساس المنشورة وبنفس معدل التعلم upstream.
- المشغّل **لا يخسر عملاً أبداً**: تجربة انقطعت تستأنف من آخر نقطة سليمة بالخطوات المتبقية فقط، وNaN يُعالج فوراً، وكل حالة موثقة في `trials_state.json` و`scheduler.log`.
- أي توقف غير طبيعي ينتج نموذجاً صالحاً من آخر نقطة سليمة — لا تضيع دقيقة تدريب دفعت ثمنها.
- سقف الميزانية يُحترم: بعده لا إطلاقات جديدة (وإيقاف حاد اختياري للجارية).
- اختيار البطل **بالأرقام** (تشابه متحدث + حفظ محتوى على بيانات معزولة) — لا بالانطباعات.
- كل ضمانات الحفظ الدائم الثلاثي (قرص شبكي + HuggingFace + tar.gz).

**ما لا يستطيع أي كود على الأرض ضمانه:**
- جودة اللهجة النهائية تعتمد على البيانات والجرعة — لكن هذه النسخة تقلّم المخاطرة: 4 وصفات متنافسة يُختار الفائز بينها بالقياس، والإيقاف المبكر يمنع حرق المال فوق الإشباع.
- الإيقاف المبكر قرار مبني على الخسارة (وكيل معقول) — الفيصل النهائي يبقى تقييم A/B، وإن أردت الجرعات الكاملة دائماً: `EARLY_STOP = False`.

---

## ما تحتاجه قبل البدء

1. حساب RunPod فيه رصيد (تقديرات الكلفة أدناه).
2. (اختياري وموصى به) حساب HuggingFace مجاني + رمز بصلاحية Write.

| السيناريو | التجارب | البطاقات | زمن التدريب | الكلفة التقريبية |
|---|---|---|---|---|
| تجربة سريعة (تجربتان) | 2 | 1 | ~35–55 د | ~$1–1.5 |
| **الشبكة الافتراضية** | 4 | 1 | ~75–115 د | **~$1.5–2.5** |
| الشبكة الافتراضية + موازاة | 4 | 2 | ~40–60 د | ~$1.7–2.5 |
| مسح أوسع (6 تجارب) | 6 | 1 | ~110–170 د | ~$2.2–3.5 |

> تُقدَّر الأزمنة على ~2 خطوة/ثانية للدفعة 6 على 5090، ومع الإيقاف المبكر غالباً أقل. **توفير ذكي:** تجهيز البيانات (الخلايا 7–13) يعمل على المعالج فقط — نفّذه بجلسة سابقة ثم استأجر البطاقة للتدريب وحده؛ الدفتر سيتخطى كل ما أُنجز تلقائياً.

</div>


<div dir="rtl">

## ١) تجهيز الحاوية على RunPod (قبل فتح الدفتر)

### أ. (موصى به) قرص شبكي دائم — Network Volume

كل ما يبقى على قرص الحاوية المحلي يُفقد عند حذف الحاوية (Terminate). القرص الشبكي يبقى **للأبد** مهما فعلت بالحاويات:

1. من لوحة [runpod.io](https://runpod.io) → **Storage** → **Network Volume** → **+ New**.
2. الحجم: **100GB** يكفي (بيانات 200 ساعة + نقاط التجارب + الحزم).
3. أنشئه في نفس المنطقة (Region) التي ستختار فيها الـ GPU.

### ب. إنشاء حاوية GPU — RTX 5090 (بطاقة واحدة أو بطاقتان)

1. **GPU Pods** → **Deploy** على **RTX 5090** — وعداد GPUs:
   - **1 بطاقة** (الخيار الافتراضي المناسب): الوضع الآلي التسلسلي — التجارب تنفَّذ واحدة تلو الأخرى تلقائياً بكل الضمانات (نقاط تفتيش، إيقاف مبكر، ميزانية).
   - **2 بطاقة** (اختياري): الوضع الموازي — تجربة على كل بطاقة وإنجاز الشبكة في نصف الزمن.
   - كلا الوضعين مدعومان بالكامل في هذا الدفتر — ولاحظ أن الفاتورة تتضاعف مع البطاقة الثانية ($2.38/ساعة).
2. القالب (Template) — **أهم خطوة على RTX 5090**:

> بطاقة 5090 من معمارية Blackwell (sm_120)، ولا تعمل مع أي قالب PyTorch 2.6 وأقدم إطلاقاً. اختر قالباً بـ **CUDA 12.8+ و PyTorch 2.7+**:

| الخيار | القالب | الحالة على 5090 |
|---|---|---|
| **القالب الذي في لوحتك (موصى به)** | **RunPod PyTorch 2.8.0** — نسخة القالب **2.1.0-2.8.0** (CUDA 12.8.1 · Ubuntu 24.04) | يعمل مباشرة — وهذا ما بُني عليه الدفتر |
| بديل ممتاز | قالب **RunPod PyTorch 2.7.1** (نسخة CUDA 12.8) | يعمل مباشرة |
| صورة مخصصة (Custom Image) | `runpod/pytorch:1.2.0-cu1281-torch280-ubuntu2404` | مثبتة ومجربة من RunPod |
| بديل مخصص | `runpod/pytorch:1.2.0-cu1281-torch271-ubuntu2404` | مثبتة ومجربة من RunPod |
| أي قالب cu121 / cu124 / cu126 (PyTorch 2.4–2.6) | مثل «RunPod PyTorch 2.4.0» القديم | لن يرى الـ GPU — **لا تختره** لـ 5090 |

> **وإن كانت قائمة «CUDA Version» في صفحة النشر تعرض 13.2 فقط:** اخترها بلا قلق — بيئة هذا الدفتر معزولة بالكامل وتحمل مكتبات CUDA 12.8 الخاصة بها داخل حزم pip (الخلية 4)، والمشغّل (الدرايفر) أحدث من كليهما، ولا نترجم أي نواة CUDA بأنفسنا. اختيار القائمة لا يؤثر على التدريب إطلاقاً.

**شبكة الأمان:** حتى لو اخترت قالباً قديماً بالخطأ، خلية البيئة (4) تبني بيئتها المعزولة بنفسها وتثبّت PyTorch 2.8.0 + CUDA 12.8 داخلها — فسيعمل التدريب على أي حال، وتفحص الخلية **كل بطاقة على حدة** بضرب مصفوفات حقيقي.

3. الإعدادات:
   - **Volume Disk**: وصّل القرص الشبكي على `/workspace`.
   - **Container Disk**: **40GB** على الأقل (بيئة مؤقتة + مخابئ + مساحة تنفس للنقاط).
4. اضغط **Deploy** ثم **Connect** → **Connect to Jupyter Lab**.
5. ارفع هذا الدفتر وافتحه.

### ج. لماذا هذا الترتيب مهم؟

| القرار | الأثر |
|---|---|
| بطاقة 5090 (أو بطاقتان) | بطاقة واحدة: تسلسل آلي كامل الضمانات؛ بطاقتان: نفس الشبكة في نصف الزمن |
| قرص شبكي على `/workspace` | بياناتك ونقاط كل التجارب وحزم النموذج تبقى بعد أي إيقاف أو حذف |
| بيئة Python معزولة على القرص الشبكي | إعادة تشغيل الحاوية لا تعيد تثبيت شيء — تستأنف خلال ثوانٍ |
| PyTorch 2.8.0 (CUDA 12.8) | يدعم Blackwell/5090 رسمياً — يُفحص على كل بطاقة قبل بدء أي عمل |

</div>


<div dir="rtl">

## ٢) خريطة الدفتر — شغّل بالترتيب من الأعلى

| # | الخلية | الوصف | المدة التقريبية |
|---|---|---|---|
| 1 | الإعدادات المركزية | كل المقابض + حفظها على القرص الدائم | ثوانٍ |
| 2 | فحص البيئة | البطاقتان / القرص / الأدوات + الدفعة لكل بطاقة | ثوانٍ |
| 3 | تبعيات النظام + uv | ffmpeg / git / مدير حزم سريع | ~دقيقة |
| 4 | بيئة Python المعزولة | venv دائم + PyTorch 2.8.0 cu128 + فحص نواة على كل بطاقة | 3–8 دقائق (مرة واحدة) |
| 5 | استنساخ المستودعين | Seed-VC (المدرب الرسمي) + رافِدَين (خط البيانات) + بصمات git | ~دقيقة |
| 6 | استخراج خط بيانات العراق | الكتالوج المدمج: 49,194 مقطعاً / 339 ساعة | ثوانٍ |
| 7 | المجموعات الصغيرة النظيفة | youssef / shiekh / safaa من HuggingFace | 5–15 دقيقة |
| 8 | بيانات YODAS العراقية | الجلب الكبير (الساعات التي حددتها) | 15–60 دقيقة |
| 9 | (اختيارية) بياناتك الخاصة | ارفع تسجيلاتك وادمجها | حسبك |
| 10 | التقطيع والترميز | كل المقاطع ← Opus 24kHz بمقاطع 1–29.5 ث | 10–40 دقيقة |
| 11 | **تقرير البيانات** | ساعات كل مصدر + مدرّج الأطوال | ثوانٍ |
| 12 | **عزل بيانات الاختبار** | استبعاد مقاطع من كل التجارب — للتقييم الأمين | ثوانٍ |
| 13 | تنزيل نقاط النموذج | الأساس + XLSR + HiFT + CAMPPlus (~2.2GB) | 5–15 دقيقة |
| 14 | **تعريف شبكة التجارب** | 4 تجارب (جرعة/دفعة/بيانات + خضوع للإيقاف المبكر) + مجموعة فرعية بروابط رمزية | ثوانٍ |
| 15 | **الإطلاق الآلي** | مشغّل خلفي: توزيع + نقاط تفتيش + إيقاف مبكر + حارس NaN + سقف ميزانية | — |
| 16 | **لوحة المراقبة الآلية** | تقدم فعلي لكل تجربة + الميزانية + نبض المشغّل + استغلال البطاقات + منحنيات | عند الرغبة |
| 17 | (اختيارية) انتظار الاكتمال | تحديث تلقائي حتى تُحسم كل التجارب (مع سبب انتهاء كل منها) | — |
| 18 | دخان أولي لكل تجربة | استدلال حقيقي بكل نموذج منجز | ~3 دقائق |
| 19 | **تقييم A/B لكل تجربة** | كل نموذج على نفس الأزواج المعزولة + جدول مقارنة | ~10–20 دقيقة |
| 20 | **اختيار البطل** | ترجيح رقمي + توّج الفائز + استماع مقارن | ثوانٍ |
| 21 | **بنشمارك البطل اللحظي** | RTF + زمن الوصول على بروفايل الإنتاج | ~3 دقائق |
| 22 | تصدير حزمة البطل | أوزان البطل + تقارير كل التجارب + بصمات | ثوانٍ |
| 23 | أرشيف tar.gz | ملف واحد جاهز للتنزيل | ثوانٍ |
| 24 | الرفع الدائم إلى HuggingFace | مستودع خاص في حسابك — للأبد | 1–3 دقائق |
| 25 | (اختيارية) rclone سحابي | Google Drive / S3 / … | — |
| 26 | (اختيارية) تحرير المساحة | حذف المصادر الخام القابلة لإعادة الجلب | — |
| 27 | دليل النشر | كيف تستخدم نموذج البطل في الإنتاج | للقراءة |

**قواعد ذهبية:**
- كل خلية **آمنة الإعادة**: تشغيلها مرة أخرى لا يكسر شيئاً ولا يكرر العمل المنجز.
- إذا انقطع كل شيء وأعدت تشغيل الحاوية والنواة: شغّل الخلايا **بالترتيب من 1** — كل منها يكتشف المنجز ويتخطاه.
- المشغّل والتدريب مستقلان عن الدفتر تماماً: أغلق المتصفح مطمئناً.
- **التجارب مستقلة**: فشل إحداها لا يمس البقية — والمشغّل يعيد المجروحة تلقائياً من آخر نقطة تفتيش سليمة.
- **بلا عمل ضائع**: أي توقف غير طبيعي (إيقاف مبكر/NaN/انهيار/ميزانية) ينتج نموذجاً صالحاً من آخر نقطة سليمة.

</div>


In [ ]:
# ============================================================ #
#  الخلية ١ — الإعدادات المركزية والأدوات المساعدة              #
#  كل مقابض التحكم في مكان واحد، وتُحفظ على القرص الدائم        #
#  (إعادة تشغيل النواة لا تفقدها — وتُعاد عند الاستئناف)         #
# ============================================================ #
import os, sys, json, subprocess, shutil, time, re
from pathlib import Path

NB_VERSION = "4.0-rtx5090-autopilot"

# ---------- 1) جذر العمل: على تخزين دائم إن وُجد ----------
# على RunPod: القرص الشبكي (Network Volume) يُوصل تلقائياً في /workspace وهو دائم.
def _pick_root():
    # IRAQI_DATA_ROOT = مسار جذر العمل كاملاً مباشرة (نفس دلالته في سكربتات المستودع)
    override = os.environ.get("IRAQI_DATA_ROOT", "").strip()
    if override:
        return Path(override)
    for cand in ("/workspace", "/root"):
        p = Path(cand)
        try:
            p.mkdir(parents=True, exist_ok=True)
            (p / ".write_test").write_text("ok"); (p / ".write_test").unlink()
            return p / "iraqi_seedvc"
        except Exception:
            continue
    return Path.home() / "iraqi_seedvc"

ROOT = _pick_root()   # /workspace/iraqi_seedvc (دائم) أو /root/iraqi_seedvc (مؤقت)
ROOT.mkdir(parents=True, exist_ok=True)
CFG_PATH  = ROOT / "nb_config.json"

# ---------- 2) المقابض الافتراضية ----------
DEFAULTS = dict(
    # البيانات
    YODAS_HOURS    = 200,    # ساعات YODAS العراقية (0 = تعطيل، "all" = الكل 339 ساعة، أو أي رقم)
    SKIP_SMALL     = False,  # تخطي المجموعات النظيفة الصغيرة الثلاث
    # التدريب
    RUN_NAME       = "iraqi_v1",  # اسم التشغيل: بادئة أسماء التجارب (<run>__t1_anchor ...) ومجلد الحزمة
    STEPS          = 3000,   # الجرعة الافتراضية — تُتجاوز لكل تجربة في شبكة التجارب (الخلية 14)
    BATCH          = "auto", # دفعة افتراضية لكل بطاقة (5090/32GB → 6 تلقائياً) — تُتجاوز لكل تجربة
    SAVE_EVERY     = 250,    # حفظ نقطة تفتيش كل N خطوة لكل تجربة (يُحتفظ بآخر نقطتين)
    TRAIN_WORKERS  = 4,      # عمال تحميل لكل تجربة (تجربتان متوازيتان × 4 = 8 ضمن 16 vCPU)
    PRICE_PER_GPU  = 1.19,   # سعر بطاقة 5090 الواحدة/ساعة — عدّاد الكلفة يضربها بعدد البطاقات الفعلي
    CHAMPION       = "auto", # "auto" = اختيار البطل تلقائياً بالأرقام (الخلية 20)، أو اسم تجربة للفرض اليدوي
    # ---- الإيقاف المبكر (جديد في 4.0 — يوقف التجربة عند استواء الخسارة ويصنع
    #      النموذج من آخر نقطة تفتيش سليمة، فلا تُحرق مالاً فوق الإشباع) ----
    EARLY_STOP     = True,   # المفتاح الرئيس للإيقاف المبكر (False = جرعة كاملة دائماً)
    ES_PATIENCE    = 600,    # كم خطوة «بلا تحسن يُذكر» قبل الإيقاف (600 ≈ 5 دقائق)
    ES_MIN_DELTA   = 0.002,  # أدنى تحسن يُعتد به: 0.2% من أفضل خسارة
    ES_MIN_STEPS   = 800,    # لا إيقاف قبل هذه الخطوة — فرصة عادلة للتجربة
    NAN_MAX_RETRIES= 2,      # مرات إعادة الإطلاق بعد انهيار NaN قبل التسليم بنقطة سليمة
    # ---- سقف الميزانية (صمام أمان مالي — 0 = تعطيل الحد) ----
    BUDGET_USD     = 15.0,   # أوقف كل شيء عند بلوغ هذه الكلفة (بالبطاقات الفعلية)
    BUDGET_HOURS   = 6.0,    # أو عند بلوغ هذا الزمن الفعلي للتدريب
    BUDGET_HARD_STOP=True,   # True = أوقف الجارية أيضاً وصنّع نماذجها؛ False = دعها تكمل طبيعياً
    # التنزيل والتجهيز
    FETCH_WORKERS  = 16,     # تنزيلات YODAS المتوازية (ارفعها إلى 32 إذا كانت الشبكة سريعة)
    PREP_WORKERS   = 0,      # عمال التقطيع والترميز (0 = تلقائي: حتى 16 نواة)
    HF_ENDPOINT    = "",     # مرآة HuggingFace إن لزم (مثال: https://hf-mirror.com) — اتركها فارغة عادة
    # التقييم
    HOLDOUT_COUNT  = 24,     # عدد مقاطع الاختبار المعزولة عن التدريب (تقييم أمين)
    EVAL_PAIRS     = 5,      # أزواج A/B في التقييم الذهبي
    EVAL_STEPS     = 25,     # خطوات التدفق أثناء التقييم (متساوية للنموذجين = مقارنة عادلة)
    # التخزين الدائم (HuggingFace Hub)
    HF_TOKEN       = "",     # رمز من hf.co/settings/tokens بصلاحية Write — للرفع الدائم (الخلية 24)
    HF_REPO_ID     = "",     # فارغ = <اسم-حسابك>/rafidain-seedvc-iraqi تلقائياً
    HF_PRIVATE     = True,   # مستودع خاص لا يراه أحد غيرك
    # التنظيف
    CLEAN_RAW_AFTER = False, # حذف المصادر الخام بعد تجهيز بيانات التدريب (يوفر مساحة — الخلية 26)
    CLEAN_CKPTS_AFTER=False, # حذف نقاط التفتيش الثقيلة للتجارب غير الفائزة بعد اختيار البطل (الخلية 26)
)

# ---------- 3) إعادة تحميل الإعدادات المحفوظة (استئناف بعد إعادة تشغيل النواة) ----------
CFG = dict(DEFAULTS)
FORCE_RESET = False   # اجعلها True مؤقتاً لإعادة كل الإعدادات إلى الافتراضية
if CFG_PATH.exists() and not FORCE_RESET:
    try:
        saved = json.loads(CFG_PATH.read_text(encoding="utf-8"))
        CFG.update({k: saved[k] for k in DEFAULTS if k in saved})
        print(f"[إعدادات] عيد تحميلها من {CFG_PATH} (عدّل ثم أعد التشغيل، أو FORCE_RESET=True للبدء من جديد)")
    except Exception as e:
        print(f"[إعدادات] تعذر قراءة المحفوظة ({e}) — نستخدم الافتراضية")
# ← عدّل أي قيمة هنا ثم أعد تشغيل الخلية، مثال:
#    CFG["YODAS_HOURS"] = 50
#    CFG["STEPS"] = 1000
CFG["NB_VERSION"] = NB_VERSION
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------- 4) المسارات الثابتة (كلها تحت الجذر الدائم) ----------
SEEDVC   = ROOT / "seed-vc"                    # مستودع Seed-VC (المدرب الأصلي)
RAFIDAIN = ROOT / "rafidain"                   # مستودع رافِدَين (الخدمة + خط البيانات)
PIPE     = ROOT / "iraqi_pipeline.py"          # خط بيانات العراق — يُستخرج في الخلية 6
DATA_DIR = ROOT / "data" / "iraqi_finetune"    # بيانات التدريب المجهزة (Opus 24kHz)
TEST_DIR = ROOT / "data" / "iraqi_test"        # مقاطع اختبار معزولة لا يدخلها التدريب
RUN_DIR  = SEEDVC / "runs" / CFG["RUN_NAME"]   # مجلد الأم — تجارب التشغيل في runs/<run>__t* (متجاورة)
EXPORTS  = ROOT / "exports"                    # حزم النموذج النهائية
VENV_PY  = ROOT / "venv" / "bin" / "python"    # بايثون البيئة المعزولة (الخلية 4)
for p in (ROOT, EXPORTS, RUN_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---------- 5) بيئة الأوامر وأداة تشغيل موحدة ----------
ENV = dict(os.environ)
ENV["PYTHONUNBUFFERED"] = "1"
ENV["HF_HUB_DISABLE_XET"] = "1"           # تنزيل HTTP قابل للاستئناف (نفس خيار سكربتات المستودع)
ENV["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
if CFG["HF_ENDPOINT"]:
    ENV["HF_ENDPOINT"] = CFG["HF_ENDPOINT"]

def sh(cmd, cwd=None, env=None, check=True, echo=True):
    # تشغيل أمر وطباعة مخرجاته لحظياً داخل الخلية — يعيد رمز الخروج
    cmd = [str(c) for c in cmd]
    if echo:
        print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None,
                            env=env or ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"فشل الأمر (رمز {rc}): {' '.join(cmd)}")
    return rc

# ---------- 6) حارس بيانات الاختبار: يُستدعى قبل التدريب لمنع تسريبها ----------
def reapply_holdout(verbose=True):
    """أعد فصل مقاطع الاختبار إن عادت لمجلد التدريب (بعد إعادة تشغيل prep مثلاً)."""
    hl = ROOT / "data" / "holdout_list.json"
    if not hl.exists():
        return 0
    moved = 0
    for rel in json.loads(hl.read_text()):
        src = DATA_DIR / rel
        if src.exists():
            TEST_DIR.mkdir(parents=True, exist_ok=True)
            src.rename(TEST_DIR / src.name)
            moved += 1
    if moved and verbose:
        print(f"[اختبار] أُعيد عزل {moved} مقطعاً إلى {TEST_DIR} (منع تسريب للتدريب)")
    return moved

# ---------- 7) ملخص ----------
print("=" * 64)
print(f"الجذر الدائم للعمل : {ROOT}")
print("=" * 64)
print(f"  • ساعات YODAS        : {CFG['YODAS_HOURS']}")
print(f"  • التدريب            : {CFG['STEPS']} خطوة، حفظ كل {CFG['SAVE_EVERY']}, دفعة {CFG['BATCH']}")
print(f"  • الكلفة             : ${CFG['PRICE_PER_GPU']:.2f}/بطاقة/ساعة — المراقبة (16) تضربها بعدد بطاقاتك الفعلي")
print(f"  • البطل              : {CFG['CHAMPION']} (اختيار تلقائي بالأرقام في الخلية 20 ما لم تفرض اسماً)")
print(f"  • التقييم            : {CFG['EVAL_PAIRS']} أزواج A/B على {CFG['HOLDOUT_COUNT']} مقطعاً معزولاً")
if CFG["EARLY_STOP"]:
    print(f"  • الإيقاف المبكر     : مفعّل — صبر {CFG['ES_PATIENCE']} خطوة بعد آخر تحسن ≥ {CFG['ES_MIN_DELTA']*100:.1f}%"
          f" ولا يوقف قبل خطوة {CFG['ES_MIN_STEPS']}")
else:
    print("  • الإيقاف المبكر     : معطّل — كل تجربة تُكمل جرعتها كاملة")
print(f"  • حارس NaN           : قتل فوري وإعادة إطلاق حتى {CFG['NAN_MAX_RETRIES']} مرات ثم التسليم بآخر نقطة سليمة")
bud = []
if CFG["BUDGET_USD"] > 0:  bud.append(f"${CFG['BUDGET_USD']:.0f}")
if CFG["BUDGET_HOURS"] > 0: bud.append(f"{CFG['BUDGET_HOURS']:.0f} ساعة فعلية")
print(f"  • سقف الميزانية      : {' أو '.join(bud) if bud else 'معطّل'}"
      f"{' — إيقاف حاد (تصنيع النماذج الجارية)' if CFG['BUDGET_HARD_STOP'] and bud else ''}")
print(f"  • اسم التشغيل        : {CFG['RUN_NAME']}")
print(f"  • رمز HuggingFace    : {'مضبوط (سيُرفع للأبد في الخلية 24)' if CFG['HF_TOKEN'] else 'غير مضبوط (اختياري — الرفع الدائم)'}")
print(f"\n[المسارات]  التدريب : {DATA_DIR}")
print(f"            الاختبار: {TEST_DIR}")
print(f"            النقاط  : {RUN_DIR}")
print(f"            التصدير : {EXPORTS}")
print(f"\nملاحظة: الرمز السري (إن وضعته) يُحفظ في {CFG_PATH} على قرصك الدائم فقط.")


In [ ]:
# ============================================================ #
#  الخلية ٢ — فحص البيئة: GPU / قرص / أدوات                    #
#  (يحدد حجم الدفعة تلقائياً ويكشف مدى ديمومة التخزين)          #
# ============================================================ #
import subprocess, os, shutil, re, json
from pathlib import Path

def gpu_query():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        return [l.strip() for l in out if l.strip()]
    except Exception:
        return []

gpus = gpu_query()
if gpus:
    for i, g in enumerate(gpus):
        parts = [p.strip() for p in g.split(",")]
        name, vram_mib, driver = parts[0], float(parts[1]), parts[2]
        print(f"[GPU {i}] {name} — ذاكرة {vram_mib/1024:.1f}GB — مشغل {driver}")
else:
    print("[GPU] لا يوجد GPU ظاهر عبر nvidia-smi — لن يعمل التدريب (تجهيز البيانات فقط).")

# ---- عدد البطاقات (التسلسل والموازاة كلاهما مدعوم رسمياً) ----
N_GPUS = len(gpus)
if N_GPUS >= 2:
    print(f"\n[وضع] {N_GPUS} بطاقات — الوضع الموازي: المشغّل (الخلية 15) يشغّل تجربة على كل بطاقة،")
    print("       والتالية تبدأ تلقائياً عند فراغ أي بطاقة.")
elif N_GPUS == 1:
    print("\n[وضع] بطاقة واحدة — الوضع الآلي التسلسلي (مدعوم بالكامل):")
    print("       المشغّل (الخلية 15) ينفّذ التجارب واحدة تلو الأخرى تلقائياً —")
    print("       نقاط التفتيش والإيقاف المبكر وحارس NaN وسقف الميزانية كلها تعمل،")
    print("       وكل تجربة تبدأ فور انتهاء سابقتها دون أي تدخل منك.")
    print("       (أضف بطاقة ثانية لاحقاً إن أردت الموازاة — نفس الدفتر كما هو)")
PRICE = float(CFG.get("PRICE_PER_GPU", 1.19))
print(f"[كلفة] {N_GPUS} × ${PRICE:.2f} = ${N_GPUS*PRICE:.2f}/ساعة (عدّاد مباشر في خلية المراقبة 16)")

# ---- كشف معمارية Blackwell (RTX 5090 = sm_120) ----
def gpu_compute_cap():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return out.splitlines()[0].strip() if out else ""
    except Exception:
        return ""

ccap = gpu_compute_cap()
first_name = gpus[0].split(",")[0].strip() if gpus else ""
is_blackwell = ccap.startswith("12.") or "5090" in first_name or "5080" in first_name or "5070" in first_name or "PRO 6000" in first_name
if ccap:
    print(f"[معمارية] compute capability {ccap} ← Blackwell (RTX 50-series)" if is_blackwell else f"[معمارية] compute capability {ccap}")
if is_blackwell:
    print("           [مهم] بطاقة Blackwell تتطلب PyTorch 2.7+ مع CUDA 12.8 (cu128).")
    print("                  خلية البيئة (4) تثبّت PyTorch 2.8.0 cu128 تلقائياً — لا شيء إضافي مطلوب منك.")
    print("                  إن رأيت لاحقاً خطأ 'no kernel image is available' فمعناه أن التدريب")
    print("                  شغّل بـ PyTorch خارج البيئة المعزولة — أعد خلية 4 وتأكد أنها أنجزت.")

def auto_batch(vram_gb):
    # دفعة آمنة لنموذج tiny (25M) مع مكوناته الثقيلة (XLSR-300m + HiFT + OpenVoice)
    # على RTX 5090 (32GB): 6 — ذاكرة أوسع بثلث من 4090 (24GB→4) تسمح بدفعة أكبر
    if vram_gb >= 70: return 10
    if vram_gb >= 30: return 6
    if vram_gb >= 24: return 5
    if vram_gb >= 20: return 4
    if vram_gb >= 15: return 2
    return 1

if gpus:
    vram_gb = float([p.strip() for p in gpus[0].split(",")][1]) / 1024
    CFG["_BATCH"] = auto_batch(vram_gb)
else:
    vram_gb = 0.0
    CFG["_BATCH"] = 2
if str(CFG["BATCH"]).lower() == "auto":
    BATCH = CFG["_BATCH"]
    print(f"[دفعة] تلقائي => BATCH = {BATCH} لكل بطاقة/تجربة (حسب {vram_gb:.0f}GB لكل GPU)")
    if gpus and BATCH >= 6:
        print("        (ذاكرة واسعة — دفعة كبيرة لكل تجربة تستهلك بيانات أكثر في كل خطوة وتسرّع التعلم)")
else:
    BATCH = int(CFG["BATCH"])
    print(f"[دفعة] يدوي  => BATCH = {BATCH}")
CFG["_BATCH"] = BATCH
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- المعالج والذاكرة ----
cpu = os.cpu_count() or 4
try:
    mem_kb = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1))
    print(f"[نظام] {cpu} نواة معالج — ذاكرة RAM ≈ {mem_kb/1e6:.1f}GB")
except Exception:
    print(f"[نظام] {cpu} نواة معالج")

# ---- القرص ونوع التخزين (ديمومة الملفات!) ----
du = shutil.disk_usage(ROOT)
print(f"[قرص] المتاح على {ROOT}: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB")
def find_mount(path):
    best, fstype, dev = "", "", ""
    try:
        for line in Path("/proc/mounts").read_text().splitlines():
            parts = line.split()
            if len(parts) >= 3 and str(path).startswith(parts[1]) and len(parts[1]) > len(best):
                best, fstype, dev = parts[1], parts[2], parts[0]
    except Exception:
        pass
    return best, fstype, dev
mnt, fstype, dev = find_mount(ROOT)
print(f"[تخزين] نقطة الوصل: {mnt or '/'} — النوع: {fstype or 'غير معروف'} — الجهاز: {dev or '?'}")
if str(ROOT).startswith("/workspace"):
    print("        [جيد] العمل تحت /workspace — إن كان هذا القرص الشبكي (Network Volume) فملفاتك")
    print("               دائمة وتبقى بعد إيقاف/حذف الحاوية (لا تحذف الـ Volume من لوحة RunPod).")
elif str(ROOT).startswith("/root") or str(ROOT).startswith("/home"):
    print("        [تحذير] لا يوجد /workspace — أنت على قرص الحاوية المحلي:")
    print("                يبقى عند الإيقاف (Stop) لكنه يضيع عند الحذف (Terminate).")
    print("                لضمان الديمومة: أنشئ Network Volume في RunPod وأعد تشغيل الدفتر،")
    print("                أو اضبط IRAQI_DATA_ROOT، أو اعتمد على الرفع الدائم (الخلية 24).")

# ---- تقدير مساحة التنزيل المطلوبة ----
need_gb = 2.2 + 6.0 + 2.6   # نقاط النماذج + بيئة Python/PyTorch + نقاط تفتيش 4 تجارب
t = str(CFG["YODAS_HOURS"]).strip().lower()
hours_label = "0"
if t not in ("0", ""):
    hours_label = "كل الكتالوج" if t in ("all", "كل") else f"{float(t):.0f} ساعة"
    # الخام ~ 3GB لكل 20 ساعة تقريباً (WebM) + المقاطع المشفرة Opus ~ 0.15GB/ساعة
    est_raw = 3.0 * (339 if t in ("all", "كل") else float(t)) / 20.0
    need_gb += est_raw + est_raw * 0.15
print(f"[تقدير] يحتاج هذا التشغيل ({hours_label} يوداس + مجموعات صغيرة) ≈ {need_gb:.0f}GB تقريباً")
if du.free / 1e9 < need_gb:
    print(f"        [تحذير] المساحة الحرة ({du.free/1e9:.1f}GB) قد لا تكفي!")
    print("                خفّض YODAS_HOURS، أو وسّع القرص الشبكي، أو فعّل CLEAN_RAW_AFTER (الخلية 26).")
else:
    print("        [جيد] المساحة الحرة كافية.")

# ---- الأدوات ----
for tool in ("ffmpeg", "ffprobe", "git", "curl", "uv"):
    print(f"[أداة] {tool:<8}: {'متوفر' if shutil.which(tool) else 'غير متوفر'}")

print("\n[فحص مبدئي] أكمل إلى الخلية 3 (التبعيات).")


In [ ]:
# ============================================================ #
#  الخلية ٣ — تبعيات النظام (ffmpeg / git) + مدير الحزم uv     #
#  (حاويات RunPod تعمل بصلاحيات root عادة — بلا sudo)          #
# ============================================================ #
import shutil, os, subprocess
from pathlib import Path

def have(x):
    return shutil.which(x) is not None

missing = [p for p in ("ffmpeg", "git") if not have(p)]
if missing:
    print(f"تثبيت حزم النظام: {', '.join(missing)} ...")
    sh(["apt-get", "update", "-qq"], check=False)
    sh(["apt-get", "install", "-y", "-qq", *missing])
for tool in ("ffmpeg", "git", "curl"):
    print(f"  {tool:<8}: {'متوفر' if have(tool) else 'غير متوفر!'}")

# ---- uv: مُسرّع تثبيت بايثون (نفس مسار سكربتات المستودع) ----
UV = shutil.which("uv")
if UV is None:
    print("\nتثبيت uv (مرة واحدة) ...")
    sh(["bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=False)
    for cand in (Path.home() / ".local" / "bin" / "uv", Path.home() / ".cargo" / "bin" / "uv"):
        if cand.exists():
            UV = str(cand)
            break
    if UV:
        os.environ["PATH"] = str(Path(UV).parent) + os.pathsep + os.environ["PATH"]
        ENV["PATH"] = os.environ["PATH"]
if UV:
    r = subprocess.run([UV, "--version"], capture_output=True, text=True)
    print(f"\nuv جاهز: {r.stdout.strip() or r.stderr.strip()}")
else:
    print("\nuv غير متوفر — سنعتمد python3 -m venv + pip كبديل في الخلية 4 (يعمل أيضاً).")


In [ ]:
# ============================================================ #
#  الخلية ٤ — بيئة Python المعزولة + PyTorch 2.8.0 (CUDA 12.8) #
#  مخصصة لبطاقات Blackwell مثل RTX 5090 (sm_120)               #
#  (تُبنى مرة واحدة على القرص الدائم وتُعاد استخدامها دائماً)    #
# ============================================================ #
# ملاحظة تقنية: PyTorch 2.6 وأقدم لا يحتوي نوًّا لمعمارية
# Blackwell (RTX 50-series) وسيفشل بخطأ «no kernel image is
# available» مهما بدا أن CUDA «متاح». لذلك نثبّت 2.8.0 cu128 —
# أول عائلة إصدارات مستقرة تدعم sm_120 رسمياً — مع بديل 2.7.1.
import subprocess, shutil, sys
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")   # تحصين: يعمل حتى بعد إعادة تشغيل النواة

TORCH_SPECS = [
    # (وصف، الأوامر المثبتة) — نجرب الأولى ثم البديل
    ("PyTorch 2.8.0 + torchaudio 2.8.0 (cu128)",
     ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"]),
    ("PyTorch 2.7.1 + torchaudio 2.7.1 (cu128 — بديل)",
     ["torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1"]),
]
IDX = ["--index-url", "https://download.pytorch.org/whl/cu128"]

def venv_ok(code):
    # فحص سريع داخل البيئة المعزولة
    if not VENV_PY.exists():
        return False
    r = subprocess.run([str(VENV_PY), "-c", code], capture_output=True, text=True, env=ENV)
    return r.returncode == 0

# ---- 1) إنشاء venv إن لم يوجد ----
if not VENV_PY.exists():
    print(f"إنشاء بيئة معزولة (Python 3.11) في {ROOT/'venv'} ...")
    if UV:
        sh([UV, "venv", "--python", "3.11", str(ROOT / "venv")])
    else:
        sh([sys.executable, "-m", "venv", str(ROOT / "venv")])

# ---- 2) PyTorch 2.8.0 cu128 (نحو 3GB — يُتخطى إن كان مثبتاً) ----
# الإصدار المقبول: 2.8.x أو 2.7.x — كلاهما يدعم Blackwell عبر عجلات cu128
if not venv_ok("import torch; assert torch.__version__.startswith(('2.8.', '2.7.'))"):
    for desc, trio in TORCH_SPECS:
        print(f"تثبيت {desc} — عدة دقائق حسب الشبكة ...")
        if UV:
            rc = sh([UV, "pip", "install", "--python", str(VENV_PY), *trio, *IDX], check=False)
        else:
            rc = sh([str(VENV_PY), "-m", "pip", "install", "-q", *trio, *IDX], check=False)
        if rc == 0 and venv_ok("import torch; assert torch.__version__.startswith(('2.8.', '2.7.'))"):
            break
    else:
        raise RuntimeError("فشل تثبيت PyTorch cu128 — تحقق من الشبكة وأعد هذه الخلية")
else:
    print("PyTorch 2.8.x/2.7.x (cu128) موجود أصلاً في البيئة — تخطّي التثبيت.")

# ---- 3) بقية الاعتماديات (مثبتة بالإصدارات نفسها التي نجح بها خط المستودع) ----
NEED = ("import torch, torchaudio, librosa, soundfile, munch, einops,"
        " transformers, pyarrow, huggingface_hub, yaml, matplotlib")
if not venv_ok(NEED):
    print("تثبيت الاعتماديات (numpy/scipy/librosa/transformers/...) ...")
    pins = ["numpy==1.26.4", "scipy==1.13.1", "librosa==0.10.2", "soundfile==0.12.1",
            "munch==4.0.0", "einops==0.8.0", "transformers==4.46.3", "pyyaml", "tqdm",
            "requests", "pyarrow==17.0.0", "huggingface_hub", "matplotlib"]
    loose = ["numpy", "scipy", "librosa", "soundfile", "munch", "einops", "transformers",
             "pyyaml", "tqdm", "requests", "pyarrow", "huggingface_hub", "matplotlib"]
    if UV:
        rc = sh([UV, "pip", "install", "--python", str(VENV_PY), *pins], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", str(VENV_PY), *loose])
    else:
        rc = sh([str(VENV_PY), "-m", "pip", "install", "-q", *pins], check=False)
        if rc != 0:
            sh([str(VENV_PY), "-m", "pip", "install", "-q", *loose])
else:
    print("الاعتماديات موجودة أصلاً — تخطّي التثبيت.")

# ---- 4) التحقق الحاسم: نواة GPU حقيقية (وليس مجرد «CUDA متاح») ----
# على Blackwell قد يقول torch.cuda.is_available() = True ثم تفشل أول عملية
# فعلية إن كانت النسخة قديمة. لذلك نشغّل ضرب مصفوفات حقيقياً هنا والآن.
GPU_PROBE = r'''
import torch, sys
print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
if not torch.cuda.is_available():
    print("FATAL: CUDA غير متاح من داخل البيئة"); sys.exit(2)
n = torch.cuda.device_count()
print(f"GPU مرئية: {n}")
ok = 0
for i in range(n):
    name = torch.cuda.get_device_name(i)
    cap = torch.cuda.get_device_capability(i)
    try:
        x = torch.rand(768, 768, device=f"cuda:{i}")
        y = (x @ x).sum().item()      # أول عملية GPU فعلية — تكشف «no kernel image» فوراً
        torch.cuda.synchronize()
        vram = torch.cuda.get_device_properties(i).total_memory/1e9
        print(f"  cuda:{i} {name} sm_{cap[0]}{cap[1]} {vram:.1f}GB — kernel OK (result={y:.1f})")
        ok += 1
    except Exception as e:
        print(f"FATAL: فشل تنفيذ نواة GPU {i}: {type(e).__name__}: {e}"); sys.exit(3)
if ok != n:
    print("FATAL: بطاقة لم تُختبر"); sys.exit(4)
'''
r = subprocess.run([str(VENV_PY), "-c", GPU_PROBE], capture_output=True, text=True, env=ENV)
print((r.stdout + r.stderr).strip())
out_all = r.stdout + r.stderr
assert r.returncode == 0 and out_all.count("kernel OK") >= 1, (
    "فشل فحص نواة GPU!\n"
    "  - إن ظهر «no kernel image is available»: نسخة PyTorch لا تدعم Blackwell — أعد هذه الخلية "
    "لتعيد التثبيت بـ cu128 (احذف مجلد venv إن لزم: !rm -rf " + str(ROOT / "venv") + ").\n"
    "  - إن ظهر «CUDA غير متاح»: تأكد أنك على حاوية GPU (RTX 5090 ×2) وليست CPU."
)
# ملاحظة: الفحص يختبر كل بطاقة على حدة — فشل بطاقة واحدة يوقف الخلية عمداً
# (لا معنى لبدء تجارب موازية وبطاقة معطوبة).
if "GPU مرئية" in out_all:
    print(f"[فحص] {out_all.count('kernel OK')} بطاقة اجتزت اختبار النواة فعلياً.")
# بصمة الإصدار في الإعدادات (تظهر في حزمة التصدير repro)
CFG["_TORCH"] = [ln for ln in out_all.splitlines() if ln.startswith("torch ")]
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[جاهز] البيئة المعزولة: {VENV_PY}")
print("[جاهز] نواة GPU مثبتة تعمل على Blackwell — انتقل إلى الخلية 5.")


In [ ]:
# ============================================================ #
#  الخلية ٥ — استنساخ المستودعات + بصمات قابلية إعادة الإنتاج   #
#  Seed-VC (المدرب الرسمي) + رافِدَين (خط البيانات العراقية)    #
#  (على القرص الدائم — يُحدثان إن وُجدا بدلاً من إعادة التنزيل) #
# ============================================================ #
import subprocess, json
from pathlib import Path

def git_clone(url, dst):
    if (dst / ".git").exists():
        print(f"تحديث {dst.name} ...")
        sh(["git", "-C", str(dst), "pull", "--ff-only"], check=False)
    else:
        print(f"استنساخ {dst.name} من {url} ...")
        sh(["git", "clone", "--depth", "1", url, str(dst)])

git_clone("https://github.com/Plachtaa/seed-vc.git", SEEDVC)
git_clone("https://github.com/Alqudimi/rafidain.git", RAFIDAIN)

# تحقق من الملفات الحرجة
for f in ("train.py", "inference.py", "configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml"):
    assert (SEEDVC / f).exists(), f"ملف مفقود في seed-vc: {f}"
assert (RAFIDAIN / "seed_vc_iraqi_ubuntu.sh").exists(), "سكربت خط البيانات مفقود في rafidain"

# ---- بصمات الإصدارات (تُحفظ مع حزمة النموذج لاحقاً — قابلية إعادة الإنتاج) ----
def git_hash(path):
    try:
        r = subprocess.run(["git", "-C", str(path), "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=30)
        return r.stdout.strip() or "?"
    except Exception:
        return "?"

repro = {
    "notebook_version": NB_VERSION,
    "seedvc_commit": git_hash(SEEDVC),
    "rafidain_commit": git_hash(RAFIDAIN),
    "captured_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(ROOT / "repro.json").write_text(json.dumps(repro, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[بصمات] seed-vc @ {repro['seedvc_commit']} | rafidain @ {repro['rafidain_commit']}")
print(f"[جاهز] seed-vc   : {SEEDVC}")
print(f"[جاهز] rafidain  : {RAFIDAIN}")


In [ ]:
# ============================================================ #
#  الخلية ٦ — استخراج خط بيانات العراق المدمج                   #
#  (الكتالوج العراقي كله مضمّن داخل سكربت المستودع نفسه:         #
#   49,194 مقطعاً / 339 ساعة / خريطة إزاحات بايت-دقيقة           #
#   داخل أرشيفات espnet/yodas3 — نستخرجه هنا كوحدة مستقلة)      #
# ============================================================ #
import re, gzip, base64, json, ast, urllib.request
from pathlib import Path

SRC_SH = RAFIDAIN / "seed_vc_iraqi_ubuntu.sh"
if not SRC_SH.exists():   # احتياط: المستودع غير منسوخ — ننزّل السكربت مباشرة من GitHub
    print("تنزيل السكربت من GitHub مباشرة ...")
    SRC_SH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Alqudimi/rafidain/main/seed_vc_iraqi_ubuntu.sh", SRC_SH)

text = SRC_SH.read_text(encoding="utf-8")
m = re.search(r"cat > \"\$ROOT/iraqi_pipeline\.py\" <<'PYEOF'\n(.*?)\nPYEOF\n", text, re.S)
assert m, "لم يُعثر على خط البيانات المدمج داخل السكربت!"
code = m.group(1)
ast.parse(code)   # تحقق من سلامة الكود المستخرج قبل الكتابة
PIPE.write_text(code, encoding="utf-8")
print(f"[خط البيانات] استُخرج إلى: {PIPE}  ({len(code):,} بايت)")

# ---- إحصاءات الكتالوج العراقي المدمج (بدون أي تنزيل) ----
def _grab(name):
    blob = re.search(name + r' = r"""(.*?)"""', code, re.S).group(1)
    return json.loads(gzip.decompress(base64.b64decode(blob)))

clips, offs, srcrank = _grab("_IQ_CLIPS_B64"), _grab("_IQ_OFFS_B64"), _grab("_IQ_SRC_B64")
hours_total = sum(c["du"] for c in clips) / 3600
resolvable = [s for s, v in offs.items() if v.get("off")]
print(f"[كتالوج اللهجة العراقية] {len(clips):,} مقطعاً | {hours_total:.0f} ساعة | "
      f"{len(resolvable):,} تسجيل مصدر قابل للجلب | {sum(offs[s]['size'] for s in resolvable)/1e9:.1f}GB لو جُلب كله")

# ---- حجم هدفك الحالي (نفس منطق الاختيار في السكربت: الجودة أولاً) ----
t = str(CFG["YODAS_HOURS"]).strip().lower()
if t in ("0", ""):
    print(f"\n[هدفك] YODAS معطّل (YODAS_HOURS=0) — الخلية 8 ستتخطاه. الأصل المتاح: {hours_total:.0f} ساعة.")
else:
    target = None if t in ("all", "كل") else float(t)
    ranked = sorted(srcrank.items(), key=lambda kv: -kv[1][2])
    chosen, h = [], 0.0
    for sid, (tar, ch, q) in ranked:
        if sid not in offs or offs[sid].get("off") is None:
            continue
        chosen.append(sid); h += ch
        if target and h >= target:
            break
    size_gb = sum(offs[s]["size"] for s in chosen) / 1e9
    label = f"{target:.0f} ساعة" if target else "الكتالوج كله"
    print(f"[هدفك] {label}: {len(chosen):,} تسجيلاً مصدراً ≈ {size_gb:.1f}GB تنزيلاً خاماً (سيُشفَّر إلى Opus للتدريب)")
print("\n[جاهز] تابع إلى خلايا البيانات (7 وما بعدها).")


<div dir="rtl">

## مرحلة البيانات — المصادر

| المصدر | ما هو | الحجم | ملاحظات |
|---|---|---|---|
| `yodas` | كلام عراقي من كتالوج YODAS-v3 الموسوم (dialect=iq): 49,194 مقطعاً / 339 ساعة عبر 2,360 تسجيلاً | حسب ساعاتك | الأكبر والأهم — جلب ذكي بإزاحات بايت من أرشيفات espnet بلا تنزيل الأرشيف كاملاً، مرتّب بترتيب الجودة |
| `youssef` | 3,867 مقطعاً عراقياً 22.05kHz مع نصوص | ~720MB | نظيف — مناسب أيضاً كمرجع استنساخ |
| `shiekh` | 827 مقطعاً لمتحدث ديني واحد 16kHz | ~640MB | يعمّق التنوع الأسلوبي |
| `safaa` | 676 عينة لصوت أنثوي عراقي (جاهزة TTS) | صغير | توازن أنثوي مهم للاستنساخ النسائي |
| `custom` | **تسجيلاتك أنت** (اختيارية) | حسبك | الخلية 9 — أعلى أثر لصوتك المستهدف |

**سياسة الجودة:** الكتالوج يرتّب المصادر بدرجة جودة محسوبة مسبقاً، والجلب يبدأ من الأفضل — أي أن «200 ساعة» تعني أفضل 200 ساعة متاحة، لا 200 عشوائية.

> **أخلاقيات:** هذه بيانات كلام عامة من أرشيفات بحثية مفتوحة. عند استنساخك صوت شخص حقيقي في الإنتاج، موافقته الصريحة الموثقة إلزامية.

</div>


In [ ]:
# ============================================================ #
#  الخلية ٧ — تنزيل المجموعات العراقية النظيفة الصغيرة          #
#  (youssef / shiekh / safaa من HuggingFace — ≈2-3GB)          #
#  قابلة للاستئناف: أعد التشغيل لتكملة ما فشل فقط              #
# ============================================================ #
if CFG["SKIP_SMALL"]:
    print("تخطي المجموعات الصغيرة (SKIP_SMALL=True) — إن أردت فعّلها من الخلية 1.")
else:
    sh([VENV_PY, PIPE, "small", "--root", ROOT])
    import os
    for name in ("youssef", "shiekh", "safaa"):
        d = ROOT / "raw" / name
        n = len(list(d.glob("*"))) if d.exists() else 0
        print(f"  {name:<8}: {n:,} ملفاً في {d}")
print("\n[جاهز] تابع إلى الخلية 8 (YODAS) — أو الخلية 9 لدمج ملفاتك الخاصة.")


In [ ]:
# ============================================================ #
#  الخلية ٨ — جلب بيانات YODAS العراقية (الكتلة الكبرى)        #
#  تنزيل مُجزَّأ متوازٍ من أرشيفات espnet/yodas3                #
#  يستغرق عادة 15-60 دقيقة (حسب الشبكة وساعاتك)                #
#  قابلاً للاستئناف 100%: أعد التشغيل ليكمل من حيث توقف         #
# ============================================================ #
import shutil

t = str(CFG["YODAS_HOURS"]).strip().lower()
if t == "0":
    print("YODAS معطّل (YODAS_HOURS=0) — تخطّي.")
else:
    free0 = shutil.disk_usage(ROOT).free / 1e9
    print(f"المساحة الحرة قبل الجلب: {free0:.1f}GB")
    sh([VENV_PY, PIPE, "yodas",
        "--root", ROOT,
        "--hours", str(CFG["YODAS_HOURS"]),
        "--workers", int(CFG["FETCH_WORKERS"])])
    n = len(list((ROOT / "raw" / "yodas_webm").glob("*.webm"))) if (ROOT / "raw" / "yodas_webm").exists() else 0
    free1 = shutil.disk_usage(ROOT).free / 1e9
    print(f"\nتسجيلات مصدر جاهزة: {n:,} ملف webm | استُهلك {free0-free1:.1f}GB | الحر الآن {free1:.1f}GB")
    state = ROOT / "raw" / "yodas_fetch_state.json"
    if state.exists():
        import json
        failed = json.loads(state.read_text()).get("failed", [])
        if failed:
            print(f"مصادر فشل جلبها: {len(failed)} — أعد هذه الخلية لإعادة محاولة المتبقي، أو تجاهل (بقية البيانات تكفي)")
print("\n[جاهز] تابع إلى الخلية 9 (اختيارية) ثم 10 (التقطيع والترميز).")


In [ ]:
# ============================================================ #
#  الخلية ٩ — (اختيارية) دمج تسجيلاتك الخاصة في التدريب         #
#  ارفع ملفاتك (wav/flac/mp3/m4a/opus/ogg) إلى:                #
#     /workspace/iraqi_seedvc/raw/custom/                      #
#  عبر متصفح ملفات Jupyter (سحب وإفلات) ثم أعد تشغيل الخلية    #
#  ستُقطَّع تلقائياً إلى مقاطع 1-29.5 ثانية وتُرمَّز Opus 24kHz   #
# ============================================================ #
import sys, os, json, time
from multiprocessing import Pool
from pathlib import Path

CUSTOM = ROOT / "raw" / "custom"
CUSTOM.mkdir(parents=True, exist_ok=True)
EXTS = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
files = [p for p in sorted(CUSTOM.iterdir()) if p.is_file() and p.suffix.lower() in EXTS]
print(f"مجلد بياناتك: {CUSTOM}")
print(f"ملفات صوتية موجودة: {len(files)}")

if not files:
    print('''
لا توجد ملفات بعد — لتفعيل ذلك:
  1) ارفع تسجيلاتك إلى المجلد أعلاه (سحب وإفلات في لوحة ملفات Jupyter)
  2) الأفضل: كلام نظيف بلا موسيقى أو ضجيج، 1-30 ثانية أو أطول (تُقطَّع تلقائياً)
  3) أعد تشغيل هذه الخلية، ثم شغّل خلية التقطيع (10) لتشمل بياناتك
(تخطَّ هذه الخلية إن لم يكن لديك بيانات خاصة — مجموعة YODAS + الصغيرة كافية تماماً)''')
else:
    sys.path.insert(0, str(ROOT))
    import iraqi_pipeline as P   # خط البيانات المستخرج في الخلية 6 — نفس منطق التقطيع المجرَّب
    import types
    a = types.SimpleNamespace(out=str(DATA_DIR))
    jobs = []
    for f in files:
        jobs.extend(P._prep_job_src(a, str(f), "custom"))
    print(f"سيُنتج عنها {len(jobs)} مقطعاً (يُتخطى الموجود منها)")
    t0, ok_n, bad_n = time.time(), 0, 0
    with Pool(processes=int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)) as pool:
        for out, dur, okk in pool.imap_unordered(P._run_prep_job, jobs, chunksize=16):
            if okk:
                ok_n += 1
            else:
                bad_n += 1
    print(f"تم: {ok_n} مقطعاً ناجحاً | {bad_n} فاشلاً في {time.time()-t0:.0f} ثانية")
    # تحديث فهرس بيانات التدريب (كما يفعل أمر prep نفسه)
    idx = DATA_DIR / "_prep_index.jsonl"
    prev = {}
    if idx.exists():
        for line in idx.read_text().splitlines():
            try:
                r = json.loads(line)
                prev[r["file"]] = r["dur"]
            except Exception:
                pass
    for j in jobs:
        prev[os.path.relpath(j[3], DATA_DIR)] = j[2]
    prev = {f: d for f, d in prev.items() if (DATA_DIR / f).exists()}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    idx.write_text("".join(json.dumps({"file": f, "dur": d}) + "\n" for f, d in sorted(prev.items())))
    print("فهرس بيانات التدريب محدّث — أكمل إلى الخلية 10.")


In [ ]:
# ============================================================ #
#  الخلية ١٠ — التقطيع والترميز: بناء بيانات التدريب النهائية   #
#  كل المقاطع -> Opus 24kHz أحادي، 1-29.5 ثانية                #
#  (يشمل YODAS + المجموعات الصغيرة + بياناتك إن وضعتها)        #
#  قابلة للاستئناف: الموجود يُتخطى                              #
# ============================================================ #
import os

workers = int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)
sh([VENV_PY, PIPE, "prep",
    "--root", ROOT,
    "--out", DATA_DIR,
    "--workers", workers])

n = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
print(f"\nمقاطع التدريب الجاهزة: {n:,} ملف Opus في {DATA_DIR}")
assert n >= 50, f"مقاطع قليلة جداً ({n}) — تأكد أن الخليتين 7 و8 أنزلتا بيانات ثم أعد المحاولة"
print("[جاهز] البيانات مبنية — تابع إلى الخلية 11 (تقرير البيانات) ثم 12 (عزل الاختبار).")


In [ ]:
# ============================================================ #
#  الخلية ١١ — تقرير بيانات التدريب (جديد في الإصدار ٢)        #
#  ساعات كل مصدر + مدرّج أطوال المقاطع + فحص جودة سريع        #
#  (تسميات الرسم بالإنجليزية لأن matplotlib لا يشكّل            #
#   الحروف العربية المتصلة بشكل صحيح)                          #
# ============================================================ #
import json
from pathlib import Path

# ---- إحصاء من فهرس التجهيز + من الملفات الفعلية ----
idx = DATA_DIR / "_prep_index.jsonl"
rows = []
if idx.exists():
    for line in idx.read_text().splitlines():
        try:
            r = json.loads(line)
            if (DATA_DIR / r["file"]).exists():
                rows.append(r)
        except Exception:
            pass

per, tot_h, tot_n = {}, 0.0, 0
for r in rows:
    src = r["file"].split("/")[0] if "/" in r["file"] else "(جذر)"
    per.setdefault(src, [0, 0.0])
    per[src][0] += 1
    per[src][1] += float(r["dur"]) / 3600
    tot_n += 1
    tot_h += float(r["dur"]) / 3600

print("=" * 52)
print("تقرير بيانات التدريب المجهزة")
print("=" * 52)
for k in sorted(per):
    print(f"  {k:<10}: {per[k][0]:6,} مقاطع | {per[k][1]:7.2f} ساعة")
print(f"  {'الإجمالي':<10}: {tot_n:6,} مقاطع | {tot_h:7.2f} ساعة")
print("=" * 52)

if tot_h < 1:
    print("[تحذير] أقل من ساعة واحدة — التدريب سيكون ضعيفاً. زد YODAS_HOURS أو أضف بياناتك.")
elif tot_h < 10:
    print("[ملاحظة] أقل من 10 ساعات — يكفي لتجربة المسار، ولجودة إنتاجية زد البيانات.")
else:
    print(f"[جيد] {tot_h:.0f} ساعة عراقية — جرعة تدريب قوية.")

# ---- مدرّج أطوال المقاطع ----
if rows:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    durs = [float(r["dur"]) for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.4), constrained_layout=True)
    ax.hist(durs, bins=60, color="#1f77b4", edgecolor="white", linewidth=0.4)
    ax.axvline(sum(durs) / len(durs), color="#d62728", lw=1.6,
               label=f"mean = {sum(durs)/len(durs):.1f}s")
    ax.set_xlabel("segment duration (s)")
    ax.set_ylabel("segments")
    ax.set_title(f"Iraqi training segments - {tot_n:,} files / {tot_h:.1f} h")
    ax.legend()
    ax.grid(alpha=0.3)
    chart = ROOT / "data" / "duration_hist.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    from IPython.display import display, Image as IPyImage
    display(IPyImage(filename=str(chart)))
    print(f"(حُفظ بشكل دائم: {chart} — سيُرفق بحزمة النموذج)")
    # فحص جودة: نسبة المقاطع خارج نطاق 1.5-29.5 ث
    bad = sum(1 for d in durs if d < 1.5 or d > 29.9)
    print(f"مقاطع خارج النطاق المثالي: {bad} ({100*bad/len(durs):.1f}%) — ستُتخطى تلقائياً في التدريب")


In [ ]:
# ============================================================ #
#  الخلية ١٢ — عزل بيانات الاختبار (جديد في الإصدار ٢)         #
#  نستبعد عدداً من المقاطع العراقية من التدريب نهائياً،         #
#  لتقييم النموذج لاحقاً على كلام لم تره عيناه إطلاقاً          #
#  (التقييم العلمي الوحيد المعتدل — وإلا فأنت تختبر الحفظ)     #
# ============================================================ #
import json, random
from pathlib import Path

TEST_DIR.mkdir(parents=True, exist_ok=True)
hl_path = ROOT / "data" / "holdout_list.json"

# 1) اختيار حتمي (نفس البذرة دائماً = نفس القائمة بين الجلسات)
idx = DATA_DIR / "_prep_index.jsonl"
rows = []
if idx.exists():
    for line in idx.read_text().splitlines():
        try:
            r = json.loads(line)
            if (DATA_DIR / r["file"]).exists():
                rows.append(r)
        except Exception:
            pass
assert rows, "لا يوجد فهرس بيانات — شغّل الخلية 10 (التقطيع) أولاً"

if hl_path.exists():
    holdout = json.loads(hl_path.read_text())
    print(f"قائمة اختبار موجودة: {len(holdout)} مقطعاً (من اختيار سابق — تُحترم كما هي)")
else:
    rng = random.Random(42)   # بذرة ثابتة: قابلية إعادة إنتاج كاملة
    # نفضل مقاطع 3-15 ثانية (مثالية كعينات تقييم: ليست قصيرة جداً ولا ضخمة)
    pool = [r for r in rows if 3.0 <= float(r["dur"]) <= 15.0]
    if len(pool) < CFG["HOLDOUT_COUNT"]:
        pool = rows
    take = min(int(CFG["HOLDOUT_COUNT"]), len(pool))
    # تنويع المصادر: نوزع الاختيار بالتساوي على المصادر المتاحة
    by_src = {}
    for r in pool:
        by_src.setdefault(r["file"].split("/")[0] if "/" in r["file"] else "root", []).append(r)
    for lst in by_src.values():
        rng.shuffle(lst)
    holdout, i = [], 0
    while len(holdout) < take:
        added = False
        for lst in by_src.values():
            if i < len(lst) and len(holdout) < take:
                holdout.append(lst[i]["file"])
                added = True
        if not added:
            break
        i += 1
    hl_path.write_text(json.dumps(holdout, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"قائمة اختبار جديدة: {len(holdout)} مقطعاً (بذرة ثابتة 42 — موزعة على المصادر)")

# 2) تنفيذ العزل الفعلي (idempotent: يعمل بأمان أي عدد من المرات)
moved = kept = 0
for rel in holdout:
    src = DATA_DIR / rel
    dst = TEST_DIR / Path(rel).name
    if src.exists():
        if dst.exists():
            src.unlink()          # موجود في جهة الاختبار أصلاً — نحذف نسخة التدريب
        else:
            src.rename(dst)
        moved += 1
    elif dst.exists():
        kept += 1

print(f"عُزل الآن: {moved} | كان معزولاً سابقاً: {kept} | إجمالي الاختبار: {len(list(TEST_DIR.glob('*.ogg')))} مقطعاً")
print(f"مجلد الاختبار: {TEST_DIR}")

# 3) تحديث إحصاءات التدريب بعد الاستبعاد (يحذف من الفهرس المداخل غير الموجودة)
sh([VENV_PY, PIPE, "stats", "--dir", DATA_DIR], check=False)

# ملاحظة: إذا أعدت تشغيل الخلية 10 (prep) لاحقاً فقد تعود بعض المقاطع لمجلد التدريب —
# لا مشكلة: خلية إطلاق التدريب (14) تعيد العزل تلقائياً قبل الانطلاق (reapply_holdout).
print("\n[جاهز] تابع إلى الخلية 13 (تنزيل نقاط النموذج).")


In [ ]:
# ============================================================ #
#  الخلية ١٣ — تنزيل نقاط النموذج الأساسية مسبقاً (≈2.2GB)      #
#  DiT_uvit_tat_xlsr_ema + XLSR-300m + HiFT + CAMPPlus +       #
#  OpenVoice — كلها في مجلد checkpoints داخل seed-vc           #
#  (يمنع تعطل بدء التدريب لاحقاً بسبب التنزيل، وقابل للاستئناف) #
# ============================================================ #
sh([VENV_PY, PIPE, "prefetch", "--repo", SEEDVC])
import shutil
ck = SEEDVC / "checkpoints"
total = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9 if ck.exists() else 0
print(f"\nحجم مخزون النقاط: {total:.2f}GB في {ck}")

# حارس: تأكيد النقطة الأم التي يبدأ منها التدريب (DiT_uvit_tat_xlsr_ema.pth)
base_ema = list(ck.rglob("DiT_uvit_tat_xlsr_ema.pth")) if ck.exists() else []
assert base_ema, "نقطة الأساس DiT_uvit_tat_xlsr_ema.pth غير موجودة بعد التنزيل — أعد هذه الخلية"
print(f"[مؤكد] نقطة الأساس جاهزة: {base_ema[0]}")
print("[جاهز] كل شيء مطلوب للتدريب موجود — تابع إلى مرحلة التدريب.")


<div dir="rtl">

## مرحلة التدريب الآلي (Autopilot) — كيف تعمل ولماذا هي صحيحة؟

**المبدأ الثابت:** كل تجربة تستدعي `train.py` **الرسمي** لمستودع Seed-VC **دون أي تعديل**، من نفس نقطة الأساس المنشورة `DiT_uvit_tat_xlsr_ema.pth`، وبنفس معدل التعلم المعتمد upstream (1e-5 المثبّت داخل المدرب). كل ما أضافه الإصدار 4.0 يدور **حول** المدرب لا داخله.

### المشغّل الآلي — سبع وظائف يؤديها عنك

| # | الوظيفة | ماذا يحدث؟ |
|---|---|---|
| 1 | **التوزيع** | بطاقة واحدة: التجارب واحدة تلو الأخرى تلقائياً. بطاقتان: تجربة على كل بطاقة والتالية تبدأ عند فراغ أي بطاقة |
| 2 | **نقاط التفتيش** | المدرب الرسمي يحفظ `DiT_epoch_*_step_*.pth` كل `SAVE_EVERY` (250) ويُبقي آخر نقطتين — والمشغّل يتحقق من سلامة كل نقطة (فحص أرشيف zip) ويحذف التالفة |
| 3 | **التصنيع** | أي توقف غير طبيعي (إيقاف مبكر/NaN/انهيار/ميزانية) ← يُبنى `ft_model.pth` **بنفس صيغة المدرب الرسمية تماماً** من آخر نقطة سليمة + ملف `model_manifest.json` يوثق كل شيء |
| 4 | **الإيقاف المبكر** | تجربة استوت خسارتها (بلا تحسن 0.2% لمدى 600 خطوة بعد خطوة 800) ← تُوقف ويُصنع نموذجها — لا حرق مال فوق الإشباع |
| 5 | **حارس NaN** | المدرب الرسمي **لا يموت** عند NaN بل يكمل تدريباً فاسداً! المشغّل يقتلها فوراً، ينظف النقاط التالفة، ويعيد الإطلاق من آخر نقطة سليمة (حتى مرتين) ثم يسلم النموذج من النقطة |
| 6 | **سقف الميزانية** | عند بلوغ $15 أو 6 ساعات فعلية: لا تجارب جديدة تُطلق، والجارية تُوقف بأمان مع تصنيع نماذجها (قابل للتعديل أو التعطيل من الخلية 1) |
| 7 | **النبض** | ملف الحالة `trials_state.json` يُكتب كل 20 ثانية — لوحة المراقبة (16) تفحص نبضه وتنبهك إن مات المشغّل |

### الاستئناف الأمين للجرعة — تفصيلة هندسية مهمة

المدرب الرسمي عند الاستئناف من نقطة تفتيش **يعيد عدّ الخطوات من صفر** (يحمّل الأوزان فقط، لا حالة المُحسِّن ولا العدّاد — هكذا صُمم upstream). لو أعدنا إطلاق التجربة بسذاج بخطواتها الكاملة ل training 3000 خطوة **إضافية** فوق ما تدربت عليه! المشغّل يعالج هذا: عند كل إعادة إطلاق يحسب **الخطوات المتبقية فقط** من جرعة التجربة الأصلية ويطلق بها — فتبقى مقارنة الوصفات عادلة والجرعة أمينة.

### الإيقاف المبكر — المفاتيح (كلها في الخلية 1)

| المفتاح | الافتراضي | المعنى |
|---|---|---|
| `EARLY_STOP` | `True` | المفتاح الرئيس — `False` = كل تجربة تكمل جرعتها كاملة |
| `ES_MIN_STEPS` | 800 | لا إيقاف قبل هذه الخطوة (فرصة عادلة) |
| `ES_PATIENCE` | 600 | كم خطوة بلا تحسن قبل الإيقاف (~5 دقائق) |
| `ES_MIN_DELTA` | 0.002 | أدنى تحسن يُعتد به (0.2% من أفضل خسارة) |
| `NAN_MAX_RETRIES` | 2 | مرات إعادة الإطلاق بعد NaN قبل التسليم بنقطة سليمة |
| `BUDGET_USD` / `BUDGET_HOURS` | 15 / 6 | سقف الكلفة أو الزمن الفعلي (0 = تعطيل) |
| `BUDGET_HARD_STOP` | `True` | أوقف الجارية أيضاً عند السقف؛ `False` = دعها تكمل طبيعياً |

**استثناء مقصود:** تجربة `t2_deep` معطّل عنها الإيقاف المبكر (`es=False` في الخلية 14) — لأن غرضها الوحيد قياس أثر **الجرعة الكاملة** (4500 خطوة) على الإشباع والنسيان؛ لو أوقفناها مبكراً لصارت نسخة من t1 وبطل الغرض من وجودها.

### الشبكة الافتراضية (4 تجارب)

| التجربة | الخطوات | الدفعة | البيانات | إيقاف مبكر؟ | ماذا تختبر؟ |
|---|---|---|---|---|---|
| `t1_anchor` | 3000 | 6 | كلها | ✓ | الوصفة المرجعية المجرّبة (نقطة الأمان) |
| `t2_deep` | 4500 | 6 | كلها | ✗ (عمداً) | هل جرعة أعمق تنفع أم تسبب نسياناً؟ |
| `t3_bs4` | 3000 | 4 | كلها | ✓ | دفعة أصغر = تحديثات أدق — هل تناسب اللهجة؟ |
| `t4_half` | 3000 | 6 | النصف | ✓ | هل كمية البيانات تصنع فرقاً عند نفس الجرعة؟ |

### الوضعان — بطاقة واحدة أو بطاقتان (نفس الدفتر)

| | بطاقة واحدة (تسلسلي) | بطاقتان (موازٍ) |
|---|---|---|
| تنفيذ الشبكة الافتراضية (13,500 خطوة) | ~75–115 دقيقة | ~40–60 دقيقة |
| الكلفة التقريبية للتدريب | ~$1.5–2.5 | ~$1.7–2.5 (بطاقتان لزمن أقصر) |
| ملاحظات | كل الضمانات تعمل كما هي | ضعف الإنتاجية في نفس النافذة الزمنية |

> مع الإيقاف المبكر غالباً تنتهي قبل هذه التقديرات — التجارب المستوية تُقطع وتُقيَّم نماذجها المصنوعة كأي نموذج آخر.

### لماذا لا نوزع تدريباً واحداً على بطاقتين (DDP)؟

لأن `train.py` الرسمي أحادي GPU بتصميمه، وتعديله يكسر ضمان «المدرب الرسمي دون تعديل» ويفتح مخاطر تزامن لا يمكننا ضبطها بالنيابة عنك. البديل الأقوى عملياً: تجارب مستقلة — وإن كانت لديك بطاقة واحدة فأنت في الوضع التسلسلي الآلي الذي يدير الجلسة كلها دون تدخل.

### ملاحظة الصدق حول الإيقاف المبكر

استواء الخسارة لا يعني دائماً توقف التحسن الإدراكي — لهذا: (1) النماذج الموقوفة تُقيَّم بمعايير A/B الموضوعية مثلها مثل المنجزة، (2) الفيصل هو القياس لا الخسارة، (3) وإن أردت الحذر الشديد: `EARLY_STOP = False` وتعود للجرعات الكاملة.

</div>


In [ ]:
# ============================================================ #
#  الخلية ١٤ — تعريف شبكة تجارب التدريب الآلي                  #
#  كل تجربة: (خطوات، دفعة، كمية بيانات) عبر train.py الرسمي    #
#  فقط — معدل التعلم مثبّت داخل المدرب (1e-5) ولا نمسه         #
# ============================================================ #
import json, os, random
from pathlib import Path

reapply_holdout()   # أمان: لا تدخل بيانات الاختبار في أي تجربة

# ---------- 1) مجموعة فرعية بروابط رمزية (لتجارب «نصف البيانات») ----------
def build_subset(frac):
    """مجلد فيه روابط رمزية لنسبة من ملفات التدريب (بذرة ثابتة = قابلية إعادة إنتاج).
    روابط رمزية فقط — لا يُستهلك قرص إضافي تقريباً، وقارئ بيانات المدرب (os.walk)
    يراها ملفات عادية."""
    sub = ROOT / "data" / f"iraqi_finetune_{int(round(frac*100))}pct"
    n_have = sum(1 for _ in sub.rglob("*.ogg")) if sub.exists() else 0
    if n_have >= 50:
        print(f"[مجموعة فرعية {int(frac*100)}%] جاهزة أصلاً: {sub} ({n_have} ملفاً)")
        return sub
    idx = DATA_DIR / "_prep_index.jsonl"
    assert idx.exists(), "لا يوجد فهرس بيانات — شغّل الخلية 10 (التقطيع) أولاً"
    files = []
    for line in idx.read_text().splitlines():
        try:
            rel = json.loads(line)["file"]
            if (DATA_DIR / rel).exists():   # المقاطع المعزولة (holdout) نُقلت أصلاً فتُتخطى تلقائياً
                files.append(rel)
        except Exception:
            pass
    assert len(files) >= 100, "بيانات قليلة جداً لبناء مجموعة فرعية"
    rng = random.Random(123)   # بذرة ثابتة: نفس الاختيار في كل جلسة
    rng.shuffle(files)
    take = files[: int(len(files) * frac)]
    for rel in take:
        src = (DATA_DIR / rel).resolve()
        dst = sub / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.symlink(src, dst)
    (sub / "_subset_manifest.json").write_text(
        json.dumps({"fraction": frac, "count": len(take), "seed": 123}, ensure_ascii=False),
        encoding="utf-8")
    print(f"[مجموعة فرعية {int(frac*100)}%] أُنشئت: {sub} ({len(take)} ملفاً بروابط رمزية)")
    return sub

SUB_CACHE = {}

# ---------- 2) شبكة التجارب — عدّل بحرية ----------
# الأبعاد المتاحة عبر واجهة train.py الرسمية فقط: الخطوات (الجرعة)،
# الدفعة، وكمية البيانات ("all" أو كسر مثل 0.5). معدل التعلم 1e-5
# مثبّت داخل المدرب الرسمي عمداً — لا نعدّل المدرب أبداً.
PRE = CFG["RUN_NAME"]   # بادئة أسماء التجارب
# es = خضوع التجربة للإيقاف المبكر (خطأ فقط عندما يكون غرض التجربة قياس الجرعة الكاملة)
TRIALS = [
    dict(name=f"{PRE}__t1_anchor", steps=3000, batch=6, data="all", es=True,
         desc="الوصفة المرجعية: كل البيانات، دفعة 6، 3000 خطوة"),
    dict(name=f"{PRE}__t2_deep",   steps=4500, batch=6, data="all", es=False,
         desc="جرعة أعمق: 4500 خطوة كاملة بلا إيقاف مبكر — قياس الإشباع/النسيان"),
    dict(name=f"{PRE}__t3_bs4",    steps=3000, batch=4, data="all", es=True,
         desc="دفعة أصغر (4): تحديثات أكثر دقة"),
    dict(name=f"{PRE}__t4_half",   steps=3000, batch=6, data=0.5, es=True,
         desc="نصف البيانات: هل الكمية تصنع فرقاً عند نفس الجرعة؟"),
]

# ---------- 3) ربط مجلد بيانات كل تجربة + فحوص ----------
for t in TRIALS:
    if t["data"] == "all":
        t["dataset_dir"] = str(DATA_DIR)
    else:
        frac = float(t["data"])
        if frac not in SUB_CACHE:
            SUB_CACHE[frac] = build_subset(frac)
        t["dataset_dir"] = str(SUB_CACHE[frac])
    t["save_every"] = int(CFG["SAVE_EVERY"])
    t["workers"] = int(CFG["TRAIN_WORKERS"])
    n_files = sum(1 for _ in Path(t["dataset_dir"]).rglob("*.ogg"))
    assert n_files >= 50, f"بيانات التجربة {t['name']} غير كافية ({n_files} ملفاً فقط)"
    t["n_files"] = n_files

names = [t["name"] for t in TRIALS]
assert len(names) == len(set(names)), "أسماء التجارب يجب أن تكون فريدة"
assert all(t["steps"] > 0 and t["batch"] >= 1 for t in TRIALS)

# ---------- 4) حفظ التعريف (يقرؤه المشغّل والمراقب والتقييم) ----------
(ROOT / "trials.json").write_text(json.dumps(TRIALS, ensure_ascii=False, indent=1),
                                  encoding="utf-8")

# ---------- 5) خطة التنفيذ: موازنة الحمل على البطاقات (الأطول أولاً) ----------
import subprocess as _sp
try:
    _out = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True, timeout=30).stdout
    n_gpus = len([l for l in _out.splitlines() if l.strip()])
except Exception:
    n_gpus = 1
order = sorted(range(len(TRIALS)), key=lambda i: -TRIALS[i]["steps"])   # الأطول أولاً
loads = [0] * n_gpus
assign = {}
for i in order:
    g = loads.index(min(loads))   # البطاقة الأقل عبئاً
    assign[TRIALS[i]["name"]] = g
    loads[g] += TRIALS[i]["steps"]

print("=" * 76)
print(f"شبكة التجارب: {len(TRIALS)} تجارب على {n_gpus} بطاقة — كلها من نفس نقطة الأساس")
print("=" * 76)
for t in TRIALS:
    es_tag = "إيقاف مبكر ✓" if (t.get("es", True) and CFG.get("EARLY_STOP", True)) else "جرعة كاملة"
    print(f"  • {t['name']}")
    print(f"      {t['desc']}")
    print(f"      خطوات {t['steps']} | دفعة {t['batch']} | {t['n_files']} ملفاً | {es_tag} | البطاقة المقترحة {assign[t['name']]}")
print("-" * 76)
print("حمل البطاقات (خطوات): " + " + ".join(f"GPU{g} = {loads[g]}" for g in range(n_gpus)))
critical = max(loads) if loads else 0
price = float(CFG.get("PRICE_PER_GPU", 1.19))
est_min = critical / (2.0 * 60)   # تقدير متحفظ: ~2 خطوة/ثانية للدفعة 6
print(f"المسار الحرج: {critical} خطوة ≈ {est_min:.0f} دقيقة تدريب فعلي")
print(f"كلفة مرحلة التدريب التقريبية: ≈ ${est_min/60 * n_gpus * price:.2f} ({n_gpus} × ${price:.2f}/ساعة)")
if CFG.get("EARLY_STOP", True):
    n_es = sum(1 for t in TRIALS if t.get("es", True))
    print(f"\n[إيقاف مبكر] مفعّل على {n_es}/{len(TRIALS)} تجارب: التجربة التي تستوي خسارتها")
    print(f"               (بلا تحسن {CFG['ES_MIN_DELTA']*100:.1f}% لمدى {CFG['ES_PATIENCE']} خطوة بعد خطوة {CFG['ES_MIN_STEPS']})")
    print("               توقف مبكراً ويُصنع نموذجها من آخر نقطة تفتيش — غالباً أسرع وأرخص من التقدير أعلاه.")
    print("               تجربة t2_deep مستثناة عمداً: غرضها قياس أثر الجرعة الكاملة نفسها.")
bud = []
if float(CFG.get("BUDGET_USD", 0)) > 0:  bud.append(f"${CFG['BUDGET_USD']:.0f}")
if float(CFG.get("BUDGET_HOURS", 0)) > 0: bud.append(f"{CFG['BUDGET_HOURS']:.0f} ساعة")
if bud:
    print(f"[ميزانية] سقف الأمان: {' أو '.join(bud)} — بعده لا تُطلق تجارب جديدة"
          + (" وتُوقف الجارية مع تصنيع نماذجها" if CFG.get("BUDGET_HARD_STOP", True) else " وتُترك الجارية تكمل"))
print("\n[جاهز] شغّل خلية الإطلاق الآلي (15) — المشغّل يتولى الباقي تلقائياً.")


In [ ]:
# ============================================================ #
#  الخلية ١٥ — الإطلاق الآلي (Autopilot): مشغّل خلفي كامل       #
#  يوزع التجارب على البطاقات (1 = تسلسل، 2 = موازاة) ويحيط       #
#  بالمدرب الرسمي بشبكة أمان: نقاط تفتيش + إيقاف مبكر + حارس     #
#  NaN + سقف ميزانية + استئناف أمين للجرعة — دون تعديل المدرب    #
# ============================================================ #
import json, os, subprocess, time, shutil
from pathlib import Path

TRIALS_JSON = ROOT / "trials.json"
assert TRIALS_JSON.exists(), "شغّل خلية التجارب (14) أولاً"
TRIALS = json.loads(TRIALS_JSON.read_text(encoding="utf-8"))
assert (SEEDVC / "train.py").exists(), "مستودع seed-vc غير موجود — شغّل الخلية 5"
assert VENV_PY.exists(), "البيئة المعزولة غير موجودة — شغّل الخلية 4 أولاً"

# ---- 1) عدد البطاقات ----
try:
    _out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=30).stdout
    N_GPUS = len([l for l in _out.splitlines() if l.strip()])
except Exception:
    N_GPUS = 0
assert N_GPUS >= 1, "لا يوجد GPU! التدريب يتطلب حاوية GPU على RunPod."
_n_data = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
assert _n_data >= 50, "بيانات تدريب غير كافية — شغّل خلايا البيانات (7-10)"
if N_GPUS == 1:
    print("[وضع] بطاقة واحدة — الوضع الآلي التسلسلي: التجارب تنفَّذ واحدة تلو الأخرى تلقائياً.")
    print("       نقاط التفتيش والإيقاف المبكر وحارس NaN والميزانية — كلها تعمل كما هي.")
else:
    print(f"[وضع] {N_GPUS} بطاقات — الوضع الموازي: تجربة على كل بطاقة، والتالية تبدأ عند فراغ أي بطاقة.")
_n_test = len(list(TEST_DIR.glob("*.ogg"))) if TEST_DIR.exists() else 0
if _n_test == 0:
    print("[تنبيه] لا توجد بيانات اختبار معزولة — يُنصح بتشغيل الخلية 12 أولاً لتقييم أمين.")

# ---- 2) سكربت المشغّل الآلي (عملية خلفية تدوم طوال التدريب) ----
SCHED = ROOT / "autopilot_scheduler.py"
SCHED.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Rafidain autopilot scheduler v4 (detached background process).

Runs the OFFICIAL seed-vc train.py verbatim - one process per GPU - and adds
around it, without touching it, a complete safety net:

  1. queue/parallel     : trials are distributed over GPUs; a freed GPU
                          starts the next pending trial (1 GPU = sequential).
  2. dose-faithful
     resume             : official train.py restarts step counting at 0 when
                          resuming a checkpoint, so a naive relaunch would
                          re-run the FULL dose. We relaunch with the
                          REMAINING steps only -> intended dose preserved.
  3. early stopping     : a trial whose EMA loss stops improving (patience
                          ES_PATIENCE after ES_MIN_STEPS, delta ES_MIN_DELTA)
                          is stopped and its model fabricated from the latest
                          intact checkpoint. Per-trial opt-out: es=False.
  4. NaN/Inf guard      : instant kill, corrupt checkpoints purged, resume
                          up to NAN_MAX_RETRIES times, then deliver the last
                          good model.
  5. fabrication        : on any non-natural stop, ft_model.pth is written in
                          the exact official net-only format from the newest
                          intact DiT checkpoint (zip integrity + NaN weights
                          verified), with model_manifest.json audit trail.
  6. budget guard       : no new launches past BUDGET_USD / BUDGET_HOURS of
                          active time; BUDGET_HARD_STOP also stops running
                          trials (fabricating their models first).
  7. heartbeat          : trials_state.json is rewritten every cycle (~20s);
                          its mtime is the pulse the monitor cell checks.

State : <ROOT>/trials_state.json      Log : <ROOT>/scheduler.log
"""
import glob, json, os, re, signal, subprocess, sys, time, zipfile
from pathlib import Path

ROOT = Path(os.environ["NB_ROOT"])
SEEDVC = ROOT / "seed-vc"
VENV_PY = Path(os.environ["NB_VENV"])
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
STATE = ROOT / "trials_state.json"
SCHED_LOG = ROOT / "scheduler.log"

CFG = {}
try:
    CFG = json.loads((ROOT / "nb_config.json").read_text(encoding="utf-8"))
except Exception:
    pass

EARLY_STOP   = bool(CFG.get("EARLY_STOP", True))
ES_PATIENCE  = int(CFG.get("ES_PATIENCE", 600))
ES_MIN_DELTA = float(CFG.get("ES_MIN_DELTA", 0.002))
ES_MIN_STEPS = int(CFG.get("ES_MIN_STEPS", 800))
NAN_MAX      = int(CFG.get("NAN_MAX_RETRIES", 2))
BUDGET_USD   = float(CFG.get("BUDGET_USD", 0) or 0)
BUDGET_HOURS = float(CFG.get("BUDGET_HOURS", 0) or 0)
BUDGET_HARD  = bool(CFG.get("BUDGET_HARD_STOP", True))
PRICE        = float(CFG.get("PRICE_PER_GPU", 1.19))

n_gpus = int(sys.argv[1]) if len(sys.argv) > 1 else 1
CYCLE = 20
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eEnaif+-]+)")
SETTLED = ("done", "stopped", "failed", "skipped")
META = {"active_secs", "prev_loop", "budget_flagged", "budget_info",
        "all_settled", "t_first_launch"}

def log(msg):
    with open(SCHED_LOG, "a") as f:
        f.write(time.strftime("[%Y-%m-%d %H:%M:%S] ") + msg + "\n")

def load_trials():
    return json.loads((ROOT / "trials.json").read_text(encoding="utf-8"))

def alive(pid):
    pid = int(pid)
    try:
        p, _ = os.waitpid(pid, os.WNOHANG)
        if p == pid:
            return False
    except (ChildProcessError, OSError):
        pass
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def new_st():
    return dict(status="pending", retries=0, nan_retries=0, pid=None, gpu=None,
                started_at=None, finished_at=None, stop_reason=None,
                done_steps=0, last_step=0, last_loss=None, best_loss=None,
                best_step=0, rates=[], ckpt_map={}, sessions=[], log_offset=0)

def load_state(trials):
    state = {}
    if STATE.exists():
        try:
            state = json.loads(STATE.read_text(encoding="utf-8"))
        except Exception:
            state = {}
    names = {t["name"] for t in trials}
    for k in list(state.keys()):          # drop entries of removed trials
        if k not in names and k not in META:
            del state[k]
    for t in trials:
        name = t["name"]
        st = state.get(name) or new_st()
        for k, v in new_st().items():     # heal schema across versions
            st.setdefault(k, v)
        if (SEEDVC / "runs" / name / "ft_model.pth").exists():
            if st["status"] not in SETTLED:
                st["status"] = "done"
                st["stop_reason"] = st["stop_reason"] or "natural"
        elif st["status"] == "running":
            if not (st.get("pid") and alive(st["pid"])):
                st["status"] = "pending"   # died -> autopilot resumes it
        elif st["status"] not in ("pending",) + SETTLED:
            st["status"] = "pending"
        state[name] = st
    state.setdefault("active_secs", 0.0)
    state.setdefault("prev_loop", None)
    state.setdefault("budget_flagged", False)
    return state

def save(state):
    tmp = STATE.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=1), encoding="utf-8")
    os.replace(tmp, STATE)                # atomic write - never a torn file

def parse_log(path, offset=0):
    """Parse console lines from a BYTE offset -> ([(step, loss)], end_offset)."""
    pts, new_off = [], offset
    if path.exists():
        with open(path, "rb") as f:
            f.seek(offset)
            raw = f.read()
            new_off = offset + len(raw)
        for ln in raw.decode("utf-8", errors="replace").splitlines():
            m = STEP_RE.search(ln)
            if m:
                try:
                    pts.append((int(m.group(2)), float(m.group(3))))
                except ValueError:
                    pass
    return pts, new_off

def ckpt_step(path):
    try:
        return int(Path(path).name.rsplit("_", 1)[1].split(".")[0])
    except Exception:
        return -1

def ckpt_files(tdir):
    return sorted(glob.glob(str(tdir / "DiT_epoch_*_step_*.pth")), key=ckpt_step)

def ckpt_ok(path):
    """Cheap integrity: a torch .pth is a zip - CRC-test it."""
    try:
        if Path(path).stat().st_size < 1_000_000:
            return False
        with zipfile.ZipFile(path) as z:
            return z.testzip() is None
    except Exception:
        return False

def purge_corrupt(tdir):
    removed = []
    for p in ckpt_files(tdir):
        if not ckpt_ok(p):
            try:
                os.remove(p)
                removed.append(Path(p).name)
            except OSError:
                pass
    return removed

def fabricate(name, reason, st):
    """ft_model.pth in the exact official net-only format from newest intact ckpt."""
    import torch
    tdir = SEEDVC / "runs" / name
    for p in reversed(ckpt_files(tdir)):
        if not ckpt_ok(p):
            log(f"FABRICATE {name}: {Path(p).name} failed integrity - skipped")
            continue
        try:
            sd = torch.load(p, map_location="cpu")
        except Exception as e:
            log(f"FABRICATE {name}: cannot load {Path(p).name}: {e}")
            continue
        net = sd.get("net")
        if not net:
            continue
        clean = True
        for mod_sd in net.values():      # net = {module: state_dict}
            if not isinstance(mod_sd, dict):
                continue
            for v in mod_sd.values():
                if hasattr(v, "is_floating_point") and v.is_floating_point() \
                        and bool(torch.isnan(v).any()):
                    clean = False
                    break
            if not clean:
                break
        if not clean:
            log(f"FABRICATE {name}: {Path(p).name} holds NaN weights - skipped")
            continue
        step = ckpt_step(p)
        eff = st.get("ckpt_map", {}).get(str(step), int(st.get("done_steps", 0)) + step)
        torch.save({"net": net}, tdir / "ft_model.pth")
        (tdir / "model_manifest.json").write_text(json.dumps({
            "source_ckpt": Path(p).name,
            "fabricated_from_step": step,
            "effective_steps": eff,
            "reason": reason,
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        }, ensure_ascii=False, indent=1), encoding="utf-8")
        log(f"FABRICATE {name}: ft_model.pth from {Path(p).name} "
            f"(effective step {eff}, reason={reason})")
        return True
    log(f"FABRICATE {name}: no intact checkpoint found")
    return False

def kill_trial(st):
    pid = st.get("pid")
    if not pid:
        return
    try:
        os.killpg(os.getpgid(int(pid)), signal.SIGTERM)
    except Exception:
        pass
    for _ in range(15):                   # let an in-flight torch.save finish
        if not alive(pid):
            return
        time.sleep(1)
    try:
        os.killpg(os.getpgid(int(pid)), signal.SIGKILL)
    except Exception:
        pass
    time.sleep(2)

def session_rate(st):
    rs = st.get("rates") or []
    if len(rs) >= 3 and rs[-1][0] - rs[0][0] >= 60:
        dt = rs[-1][0] - rs[0][0]
        ds = rs[-1][1] - rs[0][1]
        if dt > 0 and ds >= 0:
            return ds / dt
    return None

def effective(st):
    return int(st.get("done_steps", 0)) + int(st.get("last_step", 0))

# ------------------------------- main loop ------------------------------- #
trials = load_trials()
state = load_state(trials)
log(f"autopilot start: {len(trials)} trials, {n_gpus} gpu(s), "
    f"es={'on' if EARLY_STOP else 'off'} patience={ES_PATIENCE} min_steps={ES_MIN_STEPS}, "
    f"budget=${BUDGET_USD}/{BUDGET_HOURS}h hard={BUDGET_HARD} :: "
    + ", ".join(f"{t['name']}:{state[t['name']]['status']}" for t in trials))

while True:
    now = time.time()
    # -- active-time accounting (survives pod restarts: gaps capped at 90s) --
    if state.get("prev_loop"):
        state["active_secs"] = state["active_secs"] + min(now - state["prev_loop"], 90)
    state["prev_loop"] = now

    trials = load_trials()                # live grid edits are honored
    names = {t["name"] for t in trials}
    for t in trials:
        if t["name"] not in state:
            state[t["name"]] = new_st()
    for k in list(state.keys()):
        if k not in names and k not in META:
            del state[k]

    # -- budget guard ------------------------------------------------------
    active_h = state["active_secs"] / 3600.0
    cost = active_h * n_gpus * PRICE
    budget_hit = ((BUDGET_USD > 0 and cost >= BUDGET_USD)
                  or (BUDGET_HOURS > 0 and active_h >= BUDGET_HOURS))
    if budget_hit and not state.get("budget_flagged"):
        state["budget_flagged"] = True
        state["budget_info"] = {"cost_usd": round(cost, 2),
                                "active_hours": round(active_h, 2),
                                "hard_stop": BUDGET_HARD}
        log(f"BUDGET REACHED: ${cost:.2f} / {active_h:.2f}h active - "
            f"{'hard stop + fabricate' if BUDGET_HARD else 'no new launches'}")
    if budget_hit:
        any_running = any(state[t["name"]]["status"] == "running" for t in trials)
        for t in trials:
            st = state[t["name"]]
            if st["status"] == "running" and BUDGET_HARD:
                kill_trial(st)
                if fabricate(t["name"], "budget", st):
                    st.update(status="stopped", stop_reason="budget",
                              pid=None, finished_at=now)
                else:
                    st.update(status="failed", stop_reason="budget",
                              pid=None, finished_at=now)
                log(f"STOPPED {t['name']} (budget, effective {effective(st)} steps)")
            elif st["status"] == "pending" and (BUDGET_HARD or not any_running):
                st.update(status="skipped", stop_reason="budget", finished_at=now)
                log(f"SKIP {t['name']}: budget exhausted before launch")

    # -- update running trials ---------------------------------------------
    for t in trials:
        name = t["name"]
        st = state[name]
        if st["status"] != "running":
            continue
        tdir = SEEDVC / "runs" / name
        pts, _ = parse_log(tdir / "train_console.log", st.get("log_offset", 0))
        if pts:
            st["last_step"], st["last_loss"] = pts[-1]
            best_l, best_s = None, 0
            for s, l in pts:              # session-local best (delta-relative)
                if best_l is None or l < best_l * (1 - ES_MIN_DELTA):
                    best_l, best_s = l, s
            st["best_loss"], st["best_step"] = best_l, best_s
        for p in ckpt_files(tdir):        # live map: ckpt step -> effective step
            s = str(ckpt_step(p))
            if s not in st["ckpt_map"]:
                st["ckpt_map"][s] = int(st["done_steps"]) + ckpt_step(p)
        rs = st.setdefault("rates", [])
        if not rs or st["last_step"] >= rs[-1][1]:
            rs.append([now, st["last_step"]])
            del rs[:-20]

        # (a) natural completion: official trainer wrote ft_model.pth
        if (tdir / "ft_model.pth").exists():
            st.update(status="done", stop_reason="natural", pid=None, finished_at=now)
            log(f"DONE {name} (effective {effective(st)} steps)")
            continue
        # (b) NaN/Inf: official trainer does NOT die on NaN - we must kill it
        ll = st["last_loss"]
        if ll is not None and (ll != ll or abs(ll) == float("inf")):
            log(f"NaN {name}: loss={ll} at session step {st['last_step']}")
            kill_trial(st)
            removed = purge_corrupt(tdir)
            if removed:
                log(f"PURGED {name}: corrupt checkpoints {removed}")
            st["nan_retries"] += 1
            if st["nan_retries"] > NAN_MAX:
                if fabricate(name, "nan", st):
                    st.update(status="stopped", stop_reason="nan",
                              pid=None, finished_at=now)
                    log(f"STOPPED {name}: NaN x{st['nan_retries']} - "
                        f"model delivered from last good checkpoint")
                else:
                    st.update(status="failed", stop_reason="nan",
                              pid=None, finished_at=now)
                    log(f"FAILED {name}: NaN x{st['nan_retries']} and no intact ckpt")
            else:
                st.update(status="pending", pid=None)
                log(f"REQUEUE {name}: NaN retry {st['nan_retries']}/{NAN_MAX} "
                    f"(will resume from last good checkpoint)")
            continue
        # (c) process died without ft_model (crash / OOM-kill)
        if not alive(st["pid"]):
            st["retries"] += 1
            if st["retries"] <= 1:
                st.update(status="pending", pid=None)
                log(f"CRASH {name}: died without ft_model - will resume once")
            else:
                if fabricate(name, "crash", st):
                    st.update(status="stopped", stop_reason="crash",
                              pid=None, finished_at=now)
                    log(f"STOPPED {name}: crashed twice - model delivered from ckpt")
                else:
                    st.update(status="failed", stop_reason="crash",
                              pid=None, finished_at=now)
                    log(f"FAILED {name}: crashed twice and no intact ckpt")
            continue
        # (d) early stopping (per-trial opt-out via es=False)
        if (EARLY_STOP and t.get("es", True)
                and st["best_loss"] is not None
                and ckpt_files(tdir)               # something to fabricate from
                and st["last_step"] >= ES_MIN_STEPS
                and (st["last_step"] - st["best_step"]) >= ES_PATIENCE):
            log(f"EARLY-STOP {name}: no {ES_MIN_DELTA*100:.1f}% improvement for "
                f"{st['last_step'] - st['best_step']} steps "
                f"(best {st['best_loss']:.4f}@{st['best_step']}, "
                f"now step {st['last_step']})")
            kill_trial(st)
            if fabricate(name, "plateau", st):
                st.update(status="stopped", stop_reason="plateau",
                          pid=None, finished_at=now)
                log(f"STOPPED {name} (early-stop, effective {effective(st)} steps)")
            else:
                st.update(status="pending", pid=None)
                st["retries"] += 1
                if st["retries"] > 2:
                    st.update(status="failed", stop_reason="es_no_ckpt",
                              pid=None, finished_at=now)

    # -- launch pending trials on free GPUs ---------------------------------
    busy = set()
    for t in trials:
        st = state[t["name"]]
        if st["status"] == "running":
            busy.add(st["gpu"])
    for t in trials:
        name = t["name"]
        st = state[name]
        if st["status"] != "pending" or budget_hit:
            continue
        free = [g for g in range(n_gpus) if g not in busy]
        if not free:
            break
        gpu = free[0]
        tdir = SEEDVC / "runs" / name
        tdir.mkdir(parents=True, exist_ok=True)
        purge_corrupt(tdir)
        # dose-faithful remaining steps (official resume restarts count at 0)
        existing = ckpt_files(tdir)
        if existing:
            resume_step = ckpt_step(existing[-1])
            base = int(st["ckpt_map"].get(str(resume_step),
                                          int(st["done_steps"]) + resume_step))
            remaining = int(t["steps"]) - base
            if remaining < 100:
                if fabricate(name, "dose_reached", st):
                    st.update(status="stopped", stop_reason="dose_reached",
                              pid=None, finished_at=now)
                    log(f"STOPPED {name}: dose already reached "
                        f"({base}/{t['steps']} steps) - model fabricated")
                else:
                    st["retries"] += 1
                    if st["retries"] > 2:
                        st.update(status="failed", stop_reason="no_ckpt",
                                  pid=None, finished_at=now)
                continue
        else:
            remaining, base = int(t["steps"]), 0
        logf = tdir / "train_console.log"
        offset = logf.stat().st_size if logf.exists() else 0
        logf_h = open(logf, "a")
        cmd = [str(VENV_PY), "-u", "train.py",
               "--config", str(CFG_TINY),
               "--dataset-dir", str(t["dataset_dir"]),
               "--run-name", name,
               "--batch-size", str(t["batch"]),
               "--max-steps", str(remaining),
               "--max-epochs", "1000",
               "--save-every", str(t.get("save_every", 250)),
               "--num-workers", str(t.get("workers", 2)),
               "--gpu", str(gpu)]
        proc = subprocess.Popen(cmd, cwd=str(SEEDVC), stdout=logf_h,
                                stderr=subprocess.STDOUT, start_new_session=True)
        logf_h.close()
        st.update(status="running", pid=proc.pid, gpu=gpu, started_at=now,
                  finished_at=None, done_steps=base, log_offset=offset,
                  last_step=0, last_loss=None, best_loss=None, best_step=0,
                  rates=[],
                  sessions=(st.get("sessions") or [])
                  + [{"base": base, "start": now, "log_offset": offset}])
        if state.get("t_first_launch") is None:
            state["t_first_launch"] = now
        busy.add(gpu)
        log(f"LAUNCH {name} on GPU{gpu} pid {proc.pid} "
            f"remaining {remaining}/{t['steps']} steps (base {base}) "
            f"bs {t['batch']} data {t['dataset_dir']}")

    save(state)
    # -- exit when every trial is settled ------------------------------------
    if all(state[t["name"]]["status"] in SETTLED for t in trials):
        state["all_settled"] = True
        state["prev_loop"] = None
        save(state)
        log("ALL SETTLED - autopilot exit :: " + ", ".join(
            f"{t['name']}:{state[t['name']]['status']}"
            f"({state[t['name']].get('stop_reason')})" for t in trials))
        break
    time.sleep(CYCLE)
''', encoding="utf-8")

# ---- 3) تشغيل المشغّل إن لم يكن يعمل (إعادة التشغيل آمنة) ----
SPID = ROOT / "scheduler.pid"
def sched_alive():
    if not SPID.exists():
        return False
    try:
        os.kill(int(SPID.read_text().strip()), 0)
        return True
    except Exception:
        return False

if sched_alive():
    print(f"المشغّل يعمل أصلاً (PID {SPID.read_text().strip()}) — لا إطلاق مكرر.")
    print("• لتعديل شبكة التجارب (إضافة/تغيير): عدّل الخلية 14 وأعد تشغيلها — لا حاجة لأي شيء")
    print("  آخر، فالمشغّل يعيد قراءة الشبكة كل دورة (20 ثانية).")
    print("• لتعديل مفاتيح الإيقاف المبكر/الميزانية (الخلية 1): أوقف المشغّل القديم أولاً")
    print("  (!kill -TERM -$(cat scheduler.pid) من الجذر الدائم) ثم أعد هذه الخلية —")
    print("  يستأنف كل شيء من الحالة المحفوظة، وتُطبَّق المفاتيح الجديدة على ما لم يكتمل.")
else:
    env = dict(ENV)
    env["NB_ROOT"] = str(ROOT)
    env["NB_VENV"] = str(VENV_PY)
    logf = open(ROOT / "scheduler.log", "a")
    # المشغّل نفسه يعمل ببايثون البيئة المعزولة (يحتاج torch لتصنيع النماذج)
    proc = subprocess.Popen([str(VENV_PY), str(SCHED), str(N_GPUS)],
                            env=env, stdout=logf, stderr=subprocess.STDOUT,
                            start_new_session=True)   # يستقل عن النواة نهائياً
    SPID.write_text(str(proc.pid))
    time.sleep(6)
    if proc.poll() not in (None, 0):
        tail = (ROOT / "scheduler.log").read_text(errors="replace")[-2500:]
        raise RuntimeError("مات المشغّل فوراً! آخر السجل:\n" + tail)
    print(f"[انطلق] المشغّل الآلي — PID {proc.pid} يدير {len(TRIALS)} تجارب على {N_GPUS} بطاقة")
    if CFG.get("EARLY_STOP", True):
        print(f"       إيقاف مبكر: صبر {CFG['ES_PATIENCE']} خطوة بعد خطوة {CFG['ES_MIN_STEPS']} "
              f"(تحسن أدنى {CFG['ES_MIN_DELTA']*100:.1f}%)")
    bud = []
    if float(CFG.get("BUDGET_USD", 0) or 0) > 0:  bud.append(f"${CFG['BUDGET_USD']:.0f}")
    if float(CFG.get("BUDGET_HOURS", 0) or 0) > 0: bud.append(f"{CFG['BUDGET_HOURS']:.0f} ساعة")
    if bud:
        print(f"       سقف ميزانية: {' أو '.join(bud)}"
              f"{' — إيقاف حاد' if CFG.get('BUDGET_HARD_STOP', True) else ' — بلا إطلاقات جديدة بعده'}")

# ---- 4) ملخص الحالة الحالية ----
STATE = ROOT / "trials_state.json"
st = json.loads(STATE.read_text(encoding="utf-8")) if STATE.exists() else {}
status_ar = {"done": "منجزة", "running": "جارية", "pending": "منتظرة",
             "failed": "فاشلة", "stopped": "موقوفة", "skipped": "متجاوزة"}
print("\nالحالة الآن:")
for t in TRIALS:
    s = st.get(t["name"], {})
    reason = s.get("stop_reason")
    reason_s = f" ({reason})" if reason and s.get("status") in ("stopped", "failed") else ""
    print(f"  • {t['name']:<30} {status_ar.get(s.get('status', 'pending'), s.get('status', 'pending')):<9}"
          f" GPU{s.get('gpu', '-')}  pid={s.get('pid', '-')}{reason_s}")
print("\nالخطوة التالية: شغّل خلية المراقبة الآلية (16) كل فترة.")
print("المشغّل مستقل عن النواة والمتصفح تماماً — أغلقهما متى شئت.")
print("كل بطاقة تُطلق التجربة التالية تلقائياً عند فراغها — والتجربة المستوية تُوقف مبكراً")
print("ويُصنع نموذجها من آخر نقطة تفتيش سليمة — بلا أي تدخل منك.")


In [ ]:
# ============================================================ #
#  الخلية ١٦ — لوحة المراقبة الآلية (أعد تشغيلها متى شئت)      #
#  تقدم فعلي لكل تجربة + حالة الإيقاف المبكر + الميزانية       #
#  + نبض المشغّل + استغلال البطاقات والقرص + منحنيات مجتمعة    #
# ============================================================ #
import re, os, time, json, shutil
import subprocess as _sp
from pathlib import Path
from IPython.display import display, Image as IPyImage

TRIALS = json.loads((ROOT / "trials.json").read_text(encoding="utf-8")) \
    if (ROOT / "trials.json").exists() else []
assert TRIALS, "شغّل خلية التجارب (14) أولاً"
STATE = ROOT / "trials_state.json"
st_all = json.loads(STATE.read_text(encoding="utf-8")) if STATE.exists() else {}
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eEnaif+-]+)")

try:
    _out = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True, timeout=30).stdout
    N_GPUS = len([l for l in _out.splitlines() if l.strip()])
except Exception:
    N_GPUS = 1

status_ar = {"done": "منجزة", "running": "جارية", "pending": "منتظرة",
             "failed": "فاشلة", "stopped": "موقوفة", "skipped": "متجاوزة"}
reason_ar = {"natural": "اكتمال طبيعي", "plateau": "إيقاف مبكر — استواء الخسارة",
             "nan": "تسليم من آخر نقطة بعد NaN", "crash": "تسليم من آخر نقطة بعد انهيارين",
             "budget": "سقف الميزانية", "dose_reached": "الجرعة اكتملت في جلسة سابقة",
             "no_ckpt": "بلا نقطة تفتيش صالحة", "es_no_ckpt": ""}

ES_PATIENCE  = int(CFG.get("ES_PATIENCE", 600))
ES_MIN_STEPS = int(CFG.get("ES_MIN_STEPS", 800))
BUDGET_USD   = float(CFG.get("BUDGET_USD", 0) or 0)
BUDGET_HOURS = float(CFG.get("BUDGET_HOURS", 0) or 0)

# ---- نبض المشغّل: ملف الحالة يُكتب كل ~20 ثانية (mtime = النبض) ----
sched_line = ""
if st_all:
    age = time.time() - STATE.stat().st_mtime
    if st_all.get("all_settled"):
        sched_line = f"[المشغّل] أنهى مهامه وخرج بسلام — كل التجارب حُسمت"
    elif age < 120:
        sched_line = f"[المشغّل] حي — آخر نبض منذ {age:.0f} ثانية"
    else:
        sched_line = f"[المشغّل] لا نبض منذ {age/60:.1f} دقيقة! أعد تشغيل خلية الإطلاق (15) — كل شيء يستأنف بأمان"
else:
    sched_line = "[المشغّل] لا ملف حالة بعد — شغّل خلية الإطلاق (15)"

# ---- قراءة سجل كل تجربة: الجلسات تُحوَّل لخطوات فعالية متراكمة ----
def parse_between(raw, off_a, off_b):
    pts = []
    for ln in raw[off_a:off_b].decode("utf-8", errors="replace").splitlines():
        m = STEP_RE.search(ln)
        if m:
            try:
                pts.append((int(m.group(2)), float(m.group(3))))
            except ValueError:
                pass
    return pts

def rate_of(s):
    rs = s.get("rates") or []
    if len(rs) >= 3 and rs[-1][0] - rs[0][0] >= 60:
        dt = rs[-1][0] - rs[0][0]
        ds = rs[-1][1] - rs[0][1]
        if dt > 0 and ds >= 0:
            return ds / dt
    return None

info = {}
for t in TRIALS:
    name = t["name"]
    tdir = SEEDVC / "runs" / name
    logf = tdir / "train_console.log"
    s = st_all.get(name, {})
    raw = logf.read_bytes() if logf.exists() else b""
    sessions = s.get("sessions") or []
    pts = []
    if sessions:
        for i, ses in enumerate(sessions):
            off_a = int(ses.get("log_offset", 0))
            off_b = int(sessions[i + 1]["log_offset"]) if i + 1 < len(sessions) else len(raw)
            base = int(ses.get("base", 0))
            if off_a <= len(raw):
                pts += [(base + st_, l) for st_, l in parse_between(raw, off_a, off_b)]
    else:
        pts = parse_between(raw, 0, len(raw))
    status = s.get("status", "pending")
    if (tdir / "ft_model.pth").exists() and status not in ("done", "stopped"):
        status = "done"
    if status == "running":
        eff = int(s.get("done_steps", 0)) + int(s.get("last_step", 0))
        loss = s.get("last_loss")
    else:
        eff = pts[-1][0] if pts else int(s.get("done_steps", 0))
        loss = pts[-1][1] if pts else None
    manifest = None
    mf = tdir / "model_manifest.json"
    if status == "stopped" and mf.exists():
        try:
            manifest = json.loads(mf.read_text(encoding="utf-8"))
        except Exception:
            pass
    info[name] = dict(t=t, dir=tdir, pts=pts, status=status, s=s, eff=eff,
                      loss=loss, rate=rate_of(s), manifest=manifest,
                      started=s.get("started_at"))

# ---- لوحة الحالة ----
n_done  = sum(1 for i in info.values() if i["status"] == "done")
n_stop  = sum(1 for i in info.values() if i["status"] == "stopped")
n_fail  = sum(1 for i in info.values() if i["status"] == "failed")
n_skip  = sum(1 for i in info.values() if i["status"] == "skipped")
n_run   = sum(1 for i in info.values() if i["status"] == "running")
n_wait  = len(TRIALS) - n_done - n_stop - n_fail - n_skip - n_run
print("=" * 78)
print(f"التجارب: {len(TRIALS)} | منجزة {n_done} | موقوفة مبكراً {n_stop} | "
      f"جارية {n_run} | فاشلة {n_fail} | متجاوزة {n_skip} | منتظرة {n_wait}")
print("=" * 78)
for name, i in info.items():
    t = i["t"]; mx = t["steps"]; eff = i["eff"]
    pct = min(100.0 * eff / mx, 100.0) if mx else 0.0
    filled = int(28 * pct / 100)
    bar = "#" * filled + "-" * (28 - filled)
    loss_s = f" loss {i['loss']:.4f}" if i["loss"] is not None else ""
    gpu = i["s"].get("gpu")
    gpu_s = f" GPU{gpu}" if gpu is not None else ""
    rate_s = f" {i['rate']:.1f}خ/ث" if i["rate"] else ""
    line = (f"[{bar}] {pct:5.1f}% {eff:>5}/{mx}  {name.split('__')[-1]:<10}{gpu_s:<6}"
            f" {status_ar.get(i['status'], i['status'])}{loss_s}{rate_s}")
    print(line)
    if i["status"] == "running" and CFG.get("EARLY_STOP", True) and t.get("es", True):
        s = i["s"]
        if s.get("best_loss") is not None:
            stale = int(s.get("last_step", 0)) - int(s.get("best_step", 0))
            if int(s.get("last_step", 0)) < ES_MIN_STEPS:
                print(f"         └ إيقاف مبكر: يبدأ الفحص بعد خطوة {ES_MIN_STEPS} "
                      f"(وصلت {s.get('last_step', 0)})")
            else:
                print(f"         └ إيقاف مبكر: بلا تحسن منذ {stale} خطوة — "
                      f"يوقف عند {ES_PATIENCE} (بقي {max(ES_PATIENCE - stale, 0)})")
    elif i["status"] in ("stopped", "failed") and i["s"].get("stop_reason"):
        reason = reason_ar.get(i["s"]["stop_reason"], i["s"]["stop_reason"])
        extra = ""
        if i["manifest"]:
            extra = (f" — النموذج صُنع من نقطة خطوة {i['manifest'].get('effective_steps', '?')}"
                     f" ({i['manifest'].get('source_ckpt', '?')})")
        print(f"         └ السبب: {reason}{extra}")
    elif i["status"] == "skipped":
        print("         └ لم تبدأ إطلاقاً: نفد سقف الميزانية قبل دورها")

# ---- الكلفة والميزانية ----
price = float(CFG.get("PRICE_PER_GPU", 1.19))
rate = N_GPUS * price
active_h = float(st_all.get("active_secs", 0)) / 3600.0 if st_all else 0.0
cost = active_h * rate
if st_all.get("t_first_launch"):
    wall_h = (time.time() - st_all["t_first_launch"]) / 3600.0
    print(f"\n[كلفة] زمن فعلي {active_h:.2f} ساعة × {N_GPUS} بطاقة × ${price:.2f} = ${cost:.2f}")
    print(f"       (منذ انطلاق أول تجربة قبل {wall_h:.2f} ساعة — فترات توقف الحاوية لا تُحتسب)")
if BUDGET_USD > 0 and st_all:
    pct_b = min(100.0 * cost / BUDGET_USD, 100.0)
    filled = int(24 * pct_b / 100)
    bar_b = "#" * filled + "-" * (24 - filled)
    flag = "  ← بلغ السقف!" if st_all.get("budget_flagged") else ""
    print(f"[ميزانية] [{bar_b}] {pct_b:5.1f}% من ${BUDGET_USD:.0f}{flag}")
if BUDGET_HOURS > 0:
    print(f"[ميزانية] الزمن الفعلي {active_h:.2f} من {BUDGET_HOURS:.0f} ساعة")

# ---- استغلال البطاقات والقرص ----
try:
    _q = _sp.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",
                  "--format=csv,noheader,nounits"], capture_output=True, text=True,
                 timeout=30).stdout.strip().splitlines()
    for ln in _q:
        idx, util, mem_u, mem_t = [p.strip() for p in ln.split(",")]
        print(f"[GPU{idx}] استغلال {util}% — ذاكرة {mem_u}/{mem_t} MiB")
except Exception:
    pass
du = shutil.disk_usage(ROOT)
print(f"[قرص] حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB على {ROOT}")

# ---- نبض المشغّل ----
print(f"\n{sched_line}")
if (not st_all.get("all_settled")) and STATE.exists():
    if time.time() - STATE.stat().st_mtime > 120:
        print("        الحل: أعد تشغيل خلية الإطلاق (15) — المشغّل يستأنف كل شيء من حالته المحفوظة.")

# ---- حارس NaN ----
for name, i in info.items():
    if i["status"] == "running":
        ll = i["s"].get("last_loss")
        if ll is not None and (ll != ll or abs(ll) == float("inf")):
            print("\n" + "!" * 64)
            print(f"تحذير: خسارة {name} انهارت إلى NaN/Inf — المشغّل سيقتلها فوراً")
            print("ويعيد إطلاقها من آخر نقطة سليمة (حتى مرتين) ثم يسلم نموذجها من النقطة.")
            print("!" * 64)

# ---- منحنيات الخسارة مجتمعة (خطوات فعالية + علامة أفضل نقطة) ----
if any(len(i["pts"]) >= 2 for i in info.values()):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 3.8), constrained_layout=True)
    plotted = 0
    for name, i in info.items():
        if len(i["pts"]) >= 2:
            xs, ys = zip(*i["pts"])
            ax.plot(xs, ys, lw=1.1, label=name.split("__")[-1])
            b_idx = min(range(len(ys)), key=lambda k: ys[k])
            ax.scatter([xs[b_idx]], [ys[b_idx]], marker="*", s=60, zorder=5)
            plotted += 1
    ax.set_xlabel("effective step"); ax.set_ylabel("EMA loss"); ax.grid(alpha=0.3)
    ax.set_title(f"loss curves - {plotted} trials (* = best)")
    ax.legend(fontsize=8, loc="upper right", ncol=2)
    chart = ROOT / "loss_curves_all.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    display(IPyImage(filename=str(chart)))
    print(f"(حُفظ بشكل دائم: {chart})")

# ---- آخر أسطر التجارب الجارية ----
for name, i in info.items():
    if i["status"] == "running":
        logf = i["dir"] / "train_console.log"
        if logf.exists():
            lines = logf.read_text(errors="replace").splitlines()[-2:]
            print(f"\n— {name} (آخر سطرين) —")
            for ln in lines:
                print("  " + ln)

if n_done + n_stop + n_fail + n_skip == len(TRIALS):
    print(f"\n[اكتمل كل شيء] منجزة {n_done} | موقوفة مبكراً {n_stop} | فاشلة {n_fail} | متجاوزة {n_skip}")
    print("كل موقوفة لديها ft_model.pth مصنوع من آخر نقطة سليمة — تُقيَّم مثل المنجزة تماماً (الخلية 18-20).")
    print("انتقل إلى خلية التحقق (18).")
else:
    print("\nأعد هذه الخلية كل فترة — المشغّل يدير كل شيء تلقائياً دون تدخل.")


In [ ]:
# ============================================================ #
#  الخلية ١٧ — (اختيارية) انتظار اكتمال كل التجارب             #
#  تحديث تلقائي كل 30 ثانية حتى تُحسم جميعها                   #
#  (بدلها: شغّل الخلية 16 يدوياً كل فترة — نفس الغرض)          #
# ============================================================ #
import json, time
from pathlib import Path
from IPython.display import clear_output

TRIALS = json.loads((ROOT / "trials.json").read_text(encoding="utf-8"))
STATE = ROOT / "trials_state.json"
status_ar = {"done": "منجزة", "running": "جارية", "pending": "منتظرة",
             "failed": "فاشلة", "stopped": "موقوفة", "skipped": "متجاوزة"}
reason_ar = {"natural": "اكتمال طبيعي", "plateau": "إيقاف مبكر", "nan": "بعد NaN",
             "crash": "بعد انهيارين", "budget": "سقف الميزانية",
             "dose_reached": "جرعة مكتملة", "no_ckpt": "بلا نقطة", "es_no_ckpt": ""}

def snapshot():
    st = json.loads(STATE.read_text(encoding="utf-8")) if STATE.exists() else {}
    rows = []
    for t in TRIALS:
        name = t["name"]
        s = st.get(name, {})
        if (SEEDVC / "runs" / name / "ft_model.pth").exists():
            status = "done"
        else:
            status = s.get("status", "pending")
        eff = int(s.get("done_steps", 0)) + int(s.get("last_step", 0) or 0)
        reason = s.get("stop_reason")
        rows.append((name, status, s.get("gpu"), eff, t["steps"], reason))
    return rows

deadline = time.time() + 24 * 3600   # مهلة قصوى 24 ساعة
while time.time() < deadline:
    clear_output(wait=True)
    rows = snapshot()
    print("الحالة:")
    for name, status, gpu, eff, mx, reason in rows:
        gpu_s = f"GPU{gpu}" if gpu is not None else ""
        reason_s = f" [{reason_ar.get(reason, reason)}]" if reason and status in ("stopped", "failed") else ""
        print(f"  {name:<32} {status_ar.get(status, status):<9} {gpu_s:<5}"
              f" {eff:>5}/{mx} خطوة{reason_s}")
    if all(s in ("done", "stopped", "failed", "skipped") for _, s, *_ in rows):
        n_ok = sum(1 for _, s, *_ in rows if s == "done")
        n_es = sum(1 for _, s, *_ in rows if s == "stopped")
        print(f"\nاكتملت المرحلة: {n_ok} منجزة طبيعياً، {n_es} موقوفة مبكراً بنموذج مصنوع،"
              f" {len(rows) - n_ok - n_es} أخرى.")
        print("انتقل إلى خلية التحقق (18) ثم التقييم (19) واختيار البطل (20).")
        break
    time.sleep(30)
else:
    print("انتهت مهلة الانتظار (24 ساعة) — استخدم الخلية 16 للفحص اليدوي.")


<div dir="rtl">

## مرحلة التحقق والتقييم — بالأرقام لا بالانطباعات

عند اكتمال التجارب نجري أربع خطوات متدرجة **لكل تجربة لديها نموذج** (المنجزة طبيعياً والموقوفة مبكراً سواء — نماذج الأخيرة مصنوعة من آخر نقطة تفتيش سليمة وتُقيَّم بنفس المعايير تماماً):

| الخلية | ماذا تفحص | المعيار |
|---|---|---|
| 18 — دخان أولي | كل نموذج **يعمل فعلاً** ويحوّل صوتاً بجهازك (نفس المصدر والمرجع للجميع — مقارنة عادلة بالسمع) | ملف صوتي لكل تجربة بلا أخطاء |
| 19 — **تقييم A/B لكل تجربة** | كل نموذج مقابل النموذج الأساس على **نفس الأزواج العراقية المعزولة وبنفس البذرة ونفس خطوات التدفق** | صعود متوسط تشابه CAMPPlus مع بقاء تشابه XLSR |
| 20 — **اختيار البطل** | ترجيح رقمي بين كل التجارب: أعلى تحسن في تشابه المتحدث **بشرط** عدم إفساد المحتوى (هبوطه عن الأساس ≤ 0.03) | جدول مقارنة + فائز معلن |
| 21 — **بنشمارك البطل** | سرعة البطل الفائز على بروفايل الإنتاج `balanced` (كتل 180ms، 10 خطوات تدفق) | RTF أقل بكثير من 1.0 |

**كيف يقرأ غير المختص النتائج؟**
- **تشابه المتحدث (CAMPPlus):** كم يشبه الصوت الخارجُ المرجعَ العراقي (1.0 = تطابق). ارتفاعه = النموذج أمهر بالحناجر العراقية.
- **الحفاظ على المحتوى (XLSR):** هل بقيت الكلمات سليمة (1.0 = محفوظ تماماً). ثباته = الضبط لم «يأكل» الوضوح.
- **قاعدة البطل:** نرفض نموذجاً يكسب التشابه ويُفقد المحتوى — الفائز يجب أن يكسب بلا خسارة (حد التسامح 0.03 قابل للتعديل داخل الخلية 20).
- **فرض يدوي:** إن أردت تجاوز الترجيح الآلي (مثلاً فضّلت صوتاً معيناً بالسمع) اضبط `CFG["CHAMPION"] = "<اسم-التجربة>"` في الخلية 1 قبل تشغيل الخلية 20.

</div>


In [ ]:
# ============================================================ #
#  الخلية ١٨ — التحقق الأولي: استدلال حقيقي بكل تجربة منجزة     #
#  (نفس المصدر الإنجليزي + نفس المرجع العراقي لكل النماذج)      #
#  فشل تجربة واحدة غير قاتل — البقية تكمل                      #
# ============================================================ #
from IPython.display import Audio, display
from pathlib import Path
import json

TRIALS = json.loads((ROOT / "trials.json").read_text(encoding="utf-8"))
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
_state = json.loads((ROOT / "trials_state.json").read_text(encoding="utf-8")) \
    if (ROOT / "trials_state.json").exists() else {}

def model_origin(name):
    """وسم مصدر النموذج: طبيعي (أكمله المدرب) أم مصنّع من نقطة تفتيش."""
    s = _state.get(name, {})
    if (SEEDVC / "runs" / name / "model_manifest.json").exists():
        try:
            m = json.loads((SEEDVC / "runs" / name / "model_manifest.json").read_text(encoding="utf-8"))
            return f"مصنّع من نقطة الخطوة {m.get('effective_steps', '?')} ({m.get('reason', '?')})"
        except Exception:
            return "مصنّع من نقطة تفتيش"
    if s.get("stop_reason") == "natural" or s.get("status") == "done":
        return "طبيعي — أكمل المدرب جرعته كاملة"
    return "طبيعي"

# مرجع عراقي واحد مشترك لكل التجارب (نفس المدخل = مقارنة عادلة بالسمع)
ref = None
for cand_dir in (TEST_DIR, DATA_DIR / "yodas", DATA_DIR / "youssef", DATA_DIR / "safaa"):
    if cand_dir.exists():
        cands = sorted(cand_dir.glob("*.ogg"))[:1] or sorted(cand_dir.glob("*.wav"))[:1]
        if cands:
            ref = cands[0]
            break
assert ref is not None, "لا يوجد مرجع عراقي — شغّل خلايا البيانات (7-10) أولاً"
print(f"المرجع العراقي المشترك: {ref.name}\n")

results = []
for t in TRIALS:
    name = t["name"]
    FT = SEEDVC / "runs" / name / "ft_model.pth"
    if not FT.exists():
        print(f"• {name}: لا ft_model.pth (فاشلة أو لم تكتمل) — تخطّي")
        continue
    outdir = SEEDVC / "runs" / name / "verify_out"
    outdir.mkdir(parents=True, exist_ok=True)
    rc = sh([VENV_PY, "inference.py",
             "--checkpoint", FT,
             "--config", CFG_TINY,
             "--source", SEEDVC / "examples" / "source" / "source_s1.wav",
             "--target", ref,
             "--output", outdir,
             "--diffusion-steps", "16", "--fp16", "true"],
            cwd=SEEDVC, check=False)   # فشل تجربة واحدة ليس قاتلاً
    outs = sorted(outdir.glob("vc_*.wav"))
    if rc == 0 and outs:
        print(f"• {name}: نجح — {outs[0].name}  [{model_origin(name)}]")
        results.append((name, outs[0]))
    else:
        print(f"• {name}: لم ينتج صوتاً (غير قاتل — راجع السجل أعلاه)")

print("\nأنجز {0} نموذجاً — استمع للمقارنة بالسمع (نفس المصدر والمرجع لكلها):".format(len(results)))
print("لاحظ: النماذج الموقوفة مبكراً مصنوعة من آخر نقطة تفتيش سليمة — مواطنة كاملة الحقوق في التقييم.")
for name, f in results:
    print(f"\n— {name} —")
    display(Audio(filename=str(f)))

print("\n[التالي] التقييم الكمي A/B لكل تجربة (الخلية 19) — الأرقام هي الفيصل لا السمع.")


In [ ]:
# ============================================================ #
#  الخلية ١٩ — التقييم A/B الذهبي: كل تجربة مقابل الأساس       #
#                                                              #
#  نكتب مكتبة تقييم مشتركة + سكربت A/B ثم نشغّلهما لكل تجربة:  #
#   • نفس أزواج الاختبار العراقية (معزولة عن التدريب)          #
#   • نفس البذرة العشوائية ونفس خطوات التدفق للجميع            #
#   • مقاييس موضوعية: تشابه متحدث CAMPPlus + حفظ محتوى XLSR    #
#   • في النهاية: جدول مقارنة موحد بين التجارب                 #
# ============================================================ #
import json
from pathlib import Path

# ---------- المكتبة المشتركة (يستوردها سكربتا A/B والبنشمارك) ----------
LIB = ROOT / "rafidain_eval_lib.py"
LIB.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Shared evaluation utilities for the Rafidain RunPod notebook (v2).

Faithful ports, no reinvention:
  - stack loading      -> seed-vc inference.py load_models()
  - utterance convert  -> seed-vc inference.py main() (non-f0 path)
  - streaming session  -> rafidain infrastructure/seedvc/engine.py
                          (itself a port of real-time-gui.py)

Run with cwd = seed-vc repository root.
"""
import os, sys, math, argparse
import numpy as np
import torch
import torchaudio
import librosa

SEEDVC_ROOT = os.getcwd()
if SEEDVC_ROOT not in sys.path:
    sys.path.insert(0, SEEDVC_ROOT)

MODEL_FRAMES_PER_SECOND = 50   # seed-vc works on 20 ms frames
SAMPLE_RATE_IN = 16_000
CE_DIT_DIFFERENCE_S = 2.0      # XLSR vs DiT frame offset (upstream constant)


# ------------------------------------------------------------------ loading
def load_stack(checkpoint, config, fp16=True):
    """Load the tiny-XLSR stack exactly like inference.py does."""
    from inference import load_models
    args = argparse.Namespace(checkpoint=str(checkpoint), config=str(config),
                              fp16=bool(fp16), f0_condition=False)
    return load_models(args)


def find_base_ckpt():
    """Locate (or fetch) the published base checkpoint DiT_uvit_tat_xlsr_ema.pth."""
    from hf_utils import load_custom_model_from_hf
    return load_custom_model_from_hf("Plachta/Seed-VC", "DiT_uvit_tat_xlsr_ema.pth", None)


def swap_weights(model, ckpt_path):
    """Hot-swap DiT weights in-place (base -> fine-tuned or back)."""
    from modules.commons import load_checkpoint
    model, _, _, _ = load_checkpoint(model, None, str(ckpt_path),
                                     load_only_params=True, ignore_modules=[],
                                     is_distributed=False)
    for key in model:
        model[key].eval()
    model.cfm.estimator.setup_caches(max_batch_size=1, max_seq_length=8192)
    return model


# ------------------------------------------------------ utterance conversion
@torch.no_grad()
def convert_utterance(model, semantic_fn, vocoder_fn, campplus_model, to_mel,
                      src_path, ref_path, sr, diffusion_steps=25,
                      length_adjust=1.0, inference_cfg_rate=0.7, fp16=True,
                      max_src_sec=None):
    """Port of inference.py main() non-f0 path. Returns (tensor (1,T), out_sr)."""
    from inference import crossfade
    device = next(model.cfm.parameters()).device

    source_audio = librosa.load(src_path, sr=sr)[0]
    ref_audio = librosa.load(ref_path, sr=sr)[0]
    if max_src_sec:
        source_audio = source_audio[: int(sr * max_src_sec)]
    source_audio = torch.tensor(source_audio).unsqueeze(0).float().to(device)
    ref_audio = torch.tensor(ref_audio[: sr * 25]).unsqueeze(0).float().to(device)

    hop_length = 256
    max_context_window = sr // hop_length * 30
    overlap_frame_len = 16
    overlap_wave_len = overlap_frame_len * hop_length

    converted_waves_16k = torchaudio.functional.resample(source_audio, sr, 16000)
    S_alt = semantic_fn(converted_waves_16k)
    ori_waves_16k = torchaudio.functional.resample(ref_audio, sr, 16000)
    S_ori = semantic_fn(ori_waves_16k)

    mel = to_mel(source_audio.to(device).float())
    mel2 = to_mel(ref_audio.to(device).float())

    target_lengths = torch.LongTensor([int(mel.size(2) * length_adjust)]).to(mel.device)
    target2_lengths = torch.LongTensor([mel2.size(2)]).to(mel2.device)

    feat2 = torchaudio.compliance.kaldi.fbank(ori_waves_16k, num_mel_bins=80,
                                              dither=0, sample_frequency=16000)
    feat2 = feat2 - feat2.mean(dim=0, keepdim=True)
    style2 = campplus_model(feat2.unsqueeze(0))

    cond, _, _, _, _ = model.length_regulator(S_alt, ylens=target_lengths,
                                              n_quantizers=3, f0=None)
    prompt_condition, _, _, _, _ = model.length_regulator(S_ori, ylens=target2_lengths,
                                                          n_quantizers=3, f0=None)

    max_source_window = max_context_window - mel2.size(2)
    processed_frames = 0
    generated_wave_chunks = []
    while processed_frames < cond.size(1):
        chunk_cond = cond[:, processed_frames: processed_frames + max_source_window]
        is_last_chunk = processed_frames + max_source_window >= cond.size(1)
        cat_condition = torch.cat([prompt_condition, chunk_cond], dim=1)
        with torch.autocast(device_type=device.type,
                            dtype=torch.float16 if fp16 else torch.float32):
            vc_target = model.cfm.inference(cat_condition,
                                            torch.LongTensor([cat_condition.size(1)]).to(mel2.device),
                                            mel2, style2, None, diffusion_steps,
                                            inference_cfg_rate=inference_cfg_rate)
            vc_target = vc_target[:, :, mel2.size(-1):]
        vc_wave = vocoder_fn(vc_target.float()).squeeze()
        vc_wave = vc_wave[None, :]
        if processed_frames == 0:
            if is_last_chunk:
                generated_wave_chunks.append(vc_wave[0].cpu().numpy())
                break
            generated_wave_chunks.append(vc_wave[0, :-overlap_wave_len].cpu().numpy())
            previous_chunk = vc_wave[0, -overlap_wave_len:]
            processed_frames += vc_target.size(2) - overlap_frame_len
        elif is_last_chunk:
            generated_wave_chunks.append(
                crossfade(previous_chunk.cpu().numpy(), vc_wave[0].cpu().numpy(), overlap_wave_len))
            processed_frames += vc_target.size(2) - overlap_frame_len
            break
        else:
            generated_wave_chunks.append(
                crossfade(previous_chunk.cpu().numpy(),
                          vc_wave[0, :-overlap_wave_len].cpu().numpy(), overlap_wave_len))
            previous_chunk = vc_wave[0, -overlap_wave_len:]
            processed_frames += vc_target.size(2) - overlap_frame_len
    vc_wave = torch.tensor(np.concatenate(generated_wave_chunks))[None, :].float()
    return vc_wave, sr


# ------------------------------------------------------------------ metrics
def _campplus_embed(campplus_model, wav_np_16k):
    t = torch.from_numpy(np.asarray(wav_np_16k, dtype=np.float32).reshape(1, -1))
    feat = torchaudio.compliance.kaldi.fbank(t, num_mel_bins=80, dither=0,
                                             sample_frequency=16000)
    feat = feat - feat.mean(dim=0, keepdim=True)
    return campplus_model(feat.unsqueeze(0)).reshape(-1)


def speaker_similarity(campplus_model, wav_a_16k, wav_b_16k):
    """CAMPPlus cosine — timbre closeness to the reference speaker."""
    a = _campplus_embed(campplus_model, wav_a_16k)
    b = _campplus_embed(campplus_model, wav_b_16k)
    return float(torch.nn.functional.cosine_similarity(a, b, dim=0))


def _xlsr_embed(semantic_fn, wav_np_16k):
    t = torch.from_numpy(np.asarray(wav_np_16k, dtype=np.float32).reshape(1, -1))
    with torch.no_grad():
        s = semantic_fn(t)
    return s.reshape(-1, s.size(-1)).mean(dim=0)


def content_similarity(semantic_fn, wav_a_16k, wav_b_16k):
    """XLSR mean-pooled cosine — did the words survive the conversion?"""
    a = _xlsr_embed(semantic_fn, wav_a_16k)
    b = _xlsr_embed(semantic_fn, wav_b_16k)
    return float(torch.nn.functional.cosine_similarity(a, b, dim=0))


def load_16k(path, max_sec=None):
    y = librosa.load(path, sr=16000)[0]
    return y[: int(16000 * max_sec)] if max_sec else y


# ------------------------------------------------- streaming session (port)
class StreamBenchSession:
    """Port of rafidain infrastructure/seedvc/engine.py SeedVCConverterSession.

    Identical per-block mathematics to the production engine (which itself
    ports real-time-gui.py): sliding window, XLSR content features, length
    regulation, CFM at `diffusion_steps`, HiFT vocoder, equal-power crossfade.
    """

    def __init__(self, stack, profile, ref_path):
        (model, semantic_fn, f0_fn, vocoder_fn, campplus_model, to_mel, mel_fn_args) = stack
        self.model = model
        self.semantic_fn = semantic_fn
        self.vocoder_fn = vocoder_fn
        self.campplus = campplus_model
        self.to_mel = to_mel
        self.kaldi = torchaudio.compliance.kaldi
        self.sr = int(mel_fn_args["sampling_rate"])
        self.hop_length = int(mel_fn_args["hop_size"])
        self.device = next(model.cfm.parameters()).device
        self.fp16 = self.device.type == "cuda"
        self.resample_up = torchaudio.transforms.Resample(16_000, self.sr).to(self.device)
        self.profile = profile

        sr, zc = self.sr, self.sr // MODEL_FRAMES_PER_SECOND

        def q(ms):
            return int(round(ms / 1000.0 * sr / zc)) * zc

        self.block_frame = q(profile["block_ms"])
        self.block_frame_16k = 320 * self.block_frame // zc
        self.crossfade_frame = q(profile["crossfade_ms"])
        self.extra_frame = q(profile["context_left_ms"])
        self.extra_frame_right = q(profile["context_right_ms"])
        self.skip_head = self.extra_frame // zc
        self.skip_tail = self.extra_frame_right // zc
        self.return_length = (self.block_frame + self.crossfade_frame + zc) // zc
        self.ce_dit_frames = int(CE_DIT_DIFFERENCE_S * MODEL_FRAMES_PER_SECOND)

        window = (self.extra_frame + self.crossfade_frame + zc + self.block_frame
                  + self.extra_frame_right)
        self.input_wav = torch.zeros(window, device=self.device, dtype=torch.float32)
        self.input_wav_res = torch.zeros(320 * window // zc, device=self.device,
                                         dtype=torch.float32)

        self.out_len = self.return_length * sr // MODEL_FRAMES_PER_SECOND
        self.overlap = self.out_len - self.block_frame
        assert self.overlap > 0 and self.block_frame > self.overlap, "block_ms too small"
        ramp = torch.sin(0.5 * float(np.pi) * torch.linspace(
            0.0, 1.0, steps=self.overlap, device=self.device)) ** 2
        self.fade_in = ramp.float()
        self._fade_in_np = ramp.detach().cpu().numpy().astype(np.float32)
        self._fade_out_np = 1.0 - self._fade_in_np
        self._carry = None
        self._pending = np.zeros(0, dtype=np.float32)

        # ---- condition on the reference voice (same as production engine) ----
        max_ref = int(16000 * profile["max_prompt_s"])
        ref16 = load_16k(ref_path)[:max_ref]
        assert ref16.size >= 16000, "reference too short"
        ref16_t = torch.from_numpy(ref16).to(self.device)
        ref_sr_t = self.resample_up(ref16_t.reshape(1, -1)).reshape(-1)

        with torch.no_grad():
            s_ori = self.semantic_fn(ref16_t.reshape(1, -1))
            feat = self.kaldi.fbank(ref16_t.reshape(1, -1), num_mel_bins=80,
                                    dither=0, sample_frequency=SAMPLE_RATE_IN)
            feat = feat - feat.mean(dim=0, keepdim=True)
            self.style2 = self.campplus(feat.unsqueeze(0))
            self.mel2 = self.to_mel(ref_sr_t.reshape(1, -1))
            target2_lengths = torch.LongTensor([self.mel2.size(2)]).to(self.device)
            self.prompt_condition = self.model.length_regulator(
                s_ori, ylens=target2_lengths, n_quantizers=3, f0=None)[0]

    def convert_block(self, pcm):
        self._pending = np.concatenate(
            [self._pending, np.asarray(pcm, dtype=np.float32).reshape(-1)])
        outs = []
        while self._pending.size >= self.block_frame_16k:
            block = self._pending[: self.block_frame_16k]
            self._pending = self._pending[self.block_frame_16k:]
            outs.append(self._process_one(block))
        if not outs:
            return np.zeros(0, dtype=np.float32)
        return np.concatenate(outs) if len(outs) > 1 else outs[0]

    def finish(self):
        tail = np.zeros(0, dtype=np.float32)
        if self._pending.size:
            padded = np.pad(self._pending, (0, self.block_frame_16k - self._pending.size))
            self._pending = np.zeros(0, dtype=np.float32)
            tail = self._process_one(padded, final=True)
        if self._carry is not None:
            carry = self._carry.detach().cpu().numpy()
            self._carry = None
            tail = np.concatenate([tail, carry]) if tail.size else carry
        return tail

    def _process_one(self, block16k, final=False):
        device = self.device
        block16k_t = torch.from_numpy(np.ascontiguousarray(block16k)).to(device)
        with torch.no_grad():
            self.input_wav = torch.cat(
                [self.input_wav[self.block_frame:], self._up(block16k_t)]
            )[: self.input_wav.shape[0]]
            self.input_wav_res = torch.cat(
                [self.input_wav_res[self.block_frame_16k:], block16k_t]
            )[: self.input_wav_res.shape[0]]

            s_alt = self.semantic_fn(self.input_wav_res.reshape(1, -1))
            s_alt = s_alt[:, self.ce_dit_frames:]
            target_lengths = torch.LongTensor(
                [int((self.skip_head + self.return_length + self.skip_tail
                      - self.ce_dit_frames) / MODEL_FRAMES_PER_SECOND
                     * self.sr // self.hop_length)]).to(device)
            cond = self.model.length_regulator(s_alt, ylens=target_lengths,
                                               n_quantizers=3, f0=None)[0]
            cat_condition = torch.cat([self.prompt_condition, cond], dim=1)

            if self.fp16:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    vc_target = self.model.cfm.inference(
                        cat_condition,
                        torch.LongTensor([cat_condition.size(1)]).to(device),
                        self.mel2, self.style2, None,
                        self.profile["diffusion_steps"],
                        inference_cfg_rate=self.profile["inference_cfg_rate"])
            else:
                vc_target = self.model.cfm.inference(
                    cat_condition,
                    torch.LongTensor([cat_condition.size(1)]).to(device),
                    self.mel2, self.style2, None,
                    self.profile["diffusion_steps"],
                    inference_cfg_rate=self.profile["inference_cfg_rate"])
            vc_target = vc_target[:, :, self.mel2.size(-1):]
            vc_wave = self.vocoder_fn(vc_target.float()).squeeze()

            output_len = self.return_length * self.sr // MODEL_FRAMES_PER_SECOND
            tail_len = self.skip_tail * self.sr // MODEL_FRAMES_PER_SECOND
            wave = vc_wave.detach().float().cpu().numpy().reshape(-1)
            need = output_len + tail_len
            if wave.size < need:
                wave = np.pad(wave, (0, need - wave.size))
            out = (wave[-need:-tail_len] if tail_len > 0 else wave[-output_len:]).astype(np.float32)

            ov = self.overlap
            if self._carry is not None:
                carry = self._carry.detach().cpu().numpy().astype(np.float32)
                head = out[:ov] * self._fade_in_np + carry[:ov] * self._fade_out_np
                body = out[ov:] if final else out[ov: self.out_len - ov]
                emitted = np.concatenate([head, body]).astype(np.float32)
                self._carry = None if final else torch.from_numpy(out[-ov:].copy()).to(device)
            else:
                if final:
                    emitted = out.astype(np.float32)
                    self._carry = None
                else:
                    emitted = out[: self.out_len - ov].astype(np.float32)
                    self._carry = torch.from_numpy(out[-ov:].copy()).to(device)
        return emitted

    def _up(self, block16k_t):
        up = self.resample_up(block16k_t.reshape(1, -1)).reshape(-1)
        if up.shape[0] > self.block_frame:
            up = up[: self.block_frame]
        elif up.shape[0] < self.block_frame:
            up = torch.nn.functional.pad(up, (0, self.block_frame - up.shape[0]))
        return up
''', encoding="utf-8")
print(f"[مكتبة التقييم] كُتبت: {LIB}")

# ---------- سكربت A/B: الأساس مقابل المضبوط ----------
AB = ROOT / "ab_eval.py"
AB.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""A/B evaluation: published base Seed-VC vs Rafidain Iraqi fine-tune.

Fair by construction: identical held-out Iraqi pairs, identical random seed
per pair (same CFM noise), identical diffusion steps. Reports CAMPPlus
speaker similarity + XLSR content preservation per pair and averaged.
"""
import os, sys, json, glob, time, argparse, random
os.environ.setdefault("HF_HUB_CACHE", "./checkpoints/hf_cache")
sys.path.insert(0, os.environ.get("NB_ROOT", "."))

import numpy as np
import torch
import torchaudio
from rafidain_eval_lib import (load_stack, find_base_ckpt, swap_weights,
                               convert_utterance, speaker_similarity,
                               content_similarity, load_16k)

AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus")


def list_audio(d):
    out = []
    for root, _, files in os.walk(d):
        for f in sorted(files):
            if f.lower().endswith(AUDIO_EXTS):
                out.append(os.path.join(root, f))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ft-ckpt", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--test-dir", required=True)
    ap.add_argument("--fallback-dir", default=None)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--pairs", type=int, default=5)
    ap.add_argument("--diffusion-steps", type=int, default=25)
    args = ap.parse_args()

    random.seed(7)
    np.random.seed(7)
    torch.manual_seed(7)

    clips = list_audio(args.test_dir)
    in_train = False
    if len(clips) < 2 * args.pairs and args.fallback_dir:
        extra = [p for p in list_audio(args.fallback_dir)
                 if os.path.basename(p) not in {os.path.basename(c) for c in clips}]
        clips = clips + extra
        in_train = True
    assert len(clips) >= 2 * args.pairs, (
        f"need {2*args.pairs} clips, found {len(clips)}")
    if in_train:
        print("[WARN] test set too small - falling back to TRAINING clips "
              "(evaluation less honest; prefer the holdout cell)")

    random.shuffle(clips)
    sources = clips[: args.pairs]
    refs = clips[args.pairs: 2 * args.pairs]

    os.makedirs(args.out_dir, exist_ok=True)

    print("[load] base stack (DiT_uvit_tat_xlsr_ema) ...")
    base_ckpt = find_base_ckpt()
    t0 = time.time()
    stack = load_stack(base_ckpt, args.config, fp16=True)
    model, semantic_fn, f0_fn, vocoder_fn, campplus_model, to_mel, mel_fn_args = stack
    sr = int(mel_fn_args["sampling_rate"])
    print(f"[load] ready in {time.time()-t0:.0f}s (sr={sr})")

    results = {"config": {"diffusion_steps": args.diffusion_steps,
                          "pairs": args.pairs, "sr": sr,
                          "held_out": not in_train},
               "pairs": [], "summary": {}}

    def eval_model(tag):
        rows, fails = [], []
        for i, (src, ref) in enumerate(zip(sources, refs)):
            try:
                torch.manual_seed(1000 + i)      # same CFM noise for both models
                wav, out_sr = convert_utterance(
                    model, semantic_fn, vocoder_fn, campplus_model, to_mel,
                    src, ref, sr, diffusion_steps=args.diffusion_steps,
                    fp16=True, max_src_sec=12.0)
                out_name = f"{tag}_pair{i:02d}.wav"
                torchaudio.save(os.path.join(args.out_dir, out_name), wav.cpu(), out_sr)
                conv16 = wav.reshape(-1).cpu().numpy()
                conv16 = torchaudio.functional.resample(
                    torch.from_numpy(conv16), out_sr, 16000).numpy()
                sim_spk = speaker_similarity(campplus_model, conv16, load_16k(ref, 25))
                sim_cnt = content_similarity(semantic_fn, conv16, load_16k(src, 12))
                rows.append({"pair": i, "source": os.path.basename(src),
                             "reference": os.path.basename(ref),
                             "speaker_sim": round(sim_spk, 4),
                             "content_sim": round(sim_cnt, 4)})
                print(f"  [{tag}] pair {i}: speaker={sim_spk:.4f}  content={sim_cnt:.4f}")
            except Exception as exc:   # مقطع تالف لا يوقف التقييم كله
                fails.append({"pair": i, "error": str(exc)[:200]})
                print(f"  [{tag}] pair {i}: FAILED ({exc})")
        if fails:
            results.setdefault("failures", {})[tag] = fails
        assert rows, f"all pairs failed for {tag} — check audio files / logs above"
        return rows

    print("\n[eval] BASE model ...")
    results["base"] = eval_model("base")

    print("\n[swap] loading fine-tuned weights ...")
    swap_weights(model, args.ft_ckpt)

    print("\n[eval] FINE-TUNED model ...")
    results["ft"] = eval_model("ft")

    # ---- aggregate (على الأزواج المشتركة الناجحة في النموذجين — مقارنة عادلة) ----
    common = sorted({r["pair"] for r in results["base"]} &
                    {r["pair"] for r in results["ft"]})
    if len(common) < len(sources):
        print(f"[note] computing means over {len(common)} common successful pairs")
    b_spk = [r["speaker_sim"] for r in results["base"] if r["pair"] in common]
    f_spk = [r["speaker_sim"] for r in results["ft"] if r["pair"] in common]
    b_cnt = [r["content_sim"] for r in results["base"] if r["pair"] in common]
    f_cnt = [r["content_sim"] for r in results["ft"] if r["pair"] in common]
    base_spk = float(np.mean(b_spk))
    ft_spk = float(np.mean(f_spk))
    base_cnt = float(np.mean(b_cnt))
    ft_cnt = float(np.mean(f_cnt))
    results["summary"] = {
        "base_speaker_sim_mean": round(base_spk, 4),
        "ft_speaker_sim_mean": round(ft_spk, 4),
        "speaker_sim_delta": round(ft_spk - base_spk, 4),
        "base_content_sim_mean": round(base_cnt, 4),
        "ft_content_sim_mean": round(ft_cnt, 4),
        "content_sim_delta": round(ft_cnt - base_cnt, 4),
    }
    s = results["summary"]
    verdict = "IMPROVED" if (s["speaker_sim_delta"] > 0
                             and s["content_sim_delta"] > -0.02) else "NEUTRAL/REGRESSION"
    results["summary"]["verdict"] = verdict

    out_json = os.path.join(args.out_dir, "ab_eval.json")
    with open(out_json, "w", encoding="utf-8") as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2)

    print("\n" + "=" * 70)
    print("A/B SUMMARY  (CAMPPlus speaker sim | XLSR content sim)")
    print("=" * 70)
    print(f"  base model : {base_spk:.4f} | {base_cnt:.4f}")
    print(f"  fine-tuned : {ft_spk:.4f} | {ft_cnt:.4f}")
    print(f"  delta      : {s['speaker_sim_delta']:+.4f} | {s['content_sim_delta']:+.4f}")
    print(f"  verdict    : {verdict}")
    print(f"  (details)  : {out_json}")
    print("=" * 70)


if __name__ == "__main__":
    main()
''', encoding="utf-8")
print(f"[سكربت A/B] كُتب: {AB}")

# ---------- تشغيل التقييم: حلقة على كل تجربة منجزة ----------
import json
from pathlib import Path

TRIALS = json.loads((ROOT / "trials.json").read_text(encoding="utf-8"))
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

env = dict(ENV)
env["NB_ROOT"] = str(ROOT)

summaries = {}
for t in TRIALS:
    name = t["name"]
    FT = SEEDVC / "runs" / name / "ft_model.pth"
    AB_DIR = SEEDVC / "runs" / name / "ab_eval"
    if not FT.exists():
        print(f"\n=== {name}: لا ft_model.pth (فاشلة/غير مكتملة) — تخطّي ===")
        continue
    print(f"\n{'='*70}\n=== تقييم A/B: {name} ===\n{'='*70}")
    sh([VENV_PY, AB,
        "--ft-ckpt", FT,
        "--config", CFG_TINY,
        "--test-dir", TEST_DIR,
        "--fallback-dir", DATA_DIR,
        "--out-dir", AB_DIR,
        "--pairs", int(CFG["EVAL_PAIRS"]),
        "--diffusion-steps", int(CFG["EVAL_STEPS"])],
       cwd=SEEDVC, env=env)
    rep = json.loads((AB_DIR / "ab_eval.json").read_text(encoding="utf-8"))
    summaries[name] = rep["summary"]
    s_ = rep["summary"]
    held = rep["config"].get("held_out", False)
    print(f"  >> تشابه متحدث: {s_['base_speaker_sim_mean']:.4f} ← {s_['ft_speaker_sim_mean']:.4f} "
          f"({s_['speaker_sim_delta']:+.4f}) | حفظ محتوى: {s_['ft_content_sim_mean']:.4f} "
          f"| {s_['verdict']}" + ("" if held else " | (بيانات تدريب — نتيجة متفائلة نسبياً)"))

# ---------- جدول المقارنة الموحد ----------
if summaries:
    print(f"\n{'='*92}")
    print("ملخص A/B لكل التجارب — نفس الأزواج المعزولة، نفس البذرة، نفس خطوات التدفق")
    print("=" * 92)
    print(f"{'التجربة':<32} {'تشابه المضبوط':>13} {'فرق التشابه':>12} {'حفظ محتوى':>10} {'الحكم':>8}")
    best = max(summaries.items(), key=lambda kv: kv[1]["speaker_sim_delta"])[0]
    for name, s_ in sorted(summaries.items(), key=lambda kv: -kv[1]["speaker_sim_delta"]):
        mark = "  ← الأفضل" if name == best else ""
        print(f"{name:<32} {s_['ft_speaker_sim_mean']:>13.4f} {s_['speaker_sim_delta']:>+12.4f}"
              f" {s_['ft_content_sim_mean']:>10.4f} {s_['verdict']:>8}{mark}")
    print("\n[التالي] خلية اختيار البطل (20) ترجّح بينها رسمياً بالأرقام وتحدد الفائز.")
else:
    print("لا توجد تجارب منجزة لتقييمها — راجع خلية المراقبة (16).")


In [ ]:
# ============================================================ #
#  الخلية ٢٠ — اختيار البطل: الترجيح بالأرقام لا بالانطباع     #
#  المعيار: أعلى تحسن في تشابه المتحدث بشرط عدم إفساد المحتوى  #
#  (هبوطه عن الأساس ≤ 0.03) — ويمكنك فرض اختيارك يدوياً        #
# ============================================================ #
import json
from pathlib import Path
from IPython.display import Audio, display

TRIALS = json.loads((ROOT / "trials.json").read_text(encoding="utf-8"))
_st = json.loads((ROOT / "trials_state.json").read_text(encoding="utf-8")) \
    if (ROOT / "trials_state.json").exists() else {}
CONTENT_TOL = 0.03   # أقصى هبوط مقبول في حفظ المحتوى مقارنة بالأساس
reason_ar = {"natural": "اكتملت", "plateau": "إيقاف مبكر", "nan": "بعد NaN",
             "crash": "بعد انهيارين", "budget": "سقف ميزانية", "dose_reached": "جرعة مكتملة"}

rows = []
for t in TRIALS:
    j = SEEDVC / "runs" / t["name"] / "ab_eval" / "ab_eval.json"
    if j.exists():
        s = json.loads(j.read_text(encoding="utf-8"))["summary"]
        r = dict(name=t["name"])
        r.update(s)
        rows.append(r)
assert rows, "لا تقارير A/B — شغّل خلية التقييم (19) أولاً"

# ---- التأهيل: لا يقبل نموذج يكسب التشابه ويفقد المحتوى ----
for r in rows:
    r["eligible"] = (r["ft_content_sim_mean"] >= r["base_content_sim_mean"] - CONTENT_TOL)
elig = [r for r in rows if r["eligible"]]
manual = str(CFG.get("CHAMPION", "auto")).strip()

if manual not in ("", "auto"):
    champ = next((r for r in rows if r["name"] == manual), None)
    assert champ is not None, f"CHAMPION={manual} غير موجود ضمن التجارب المنجزة"
    decided = "فرض يدوي من CFG['CHAMPION']"
else:
    assert elig, ("لا تجربة مؤهلة: كلها خسرت محتوى كثيراً — "
                  "خفّض الخطوات في الخلية 14، أعد الإطلاق (15)، ثم أعد التقييم (19)")
    champ = max(elig, key=lambda r: r["speaker_sim_delta"])
    decided = "ترتيب رقمي تلقائي (أعلى تحسن تشابه ضمن المؤهلات)"

# ---- الحفظ والعرض ----
(ROOT / "champion.json").write_text(json.dumps(
    {"name": champ["name"], "decided_by": decided, "metrics": champ,
     "content_tolerance": CONTENT_TOL,
     "all_trials": sorted(rows, key=lambda r: -r["speaker_sim_delta"])},
    ensure_ascii=False, indent=1), encoding="utf-8")

verdict_ar = {"IMPROVED": "تحسّن", "NEUTRAL": "محايد", "DEGRADED": "تراجع"}
print("=" * 100)
print(f"البطل: {champ['name']}   ({decided})")
print("=" * 100)
print(f"{'التجربة':<24} {'فرق التشابه':>11} {'تشابه المضبوط':>13} {'حفظ محتوى':>10} {'مؤهلة':>6} {'كيف انتهت':>13}")
for r in sorted(rows, key=lambda r: -r["speaker_sim_delta"]):
    star = "  ← البطل" if r["name"] == champ["name"] else ""
    s_ = _st.get(r["name"], {})
    end_s = reason_ar.get(s_.get("stop_reason") or "natural", "اكتملت")
    print(f"{r['name']:<24} {r['speaker_sim_delta']:>+11.4f} {r['ft_speaker_sim_mean']:>13.4f}"
          f" {r['ft_content_sim_mean']:>10.4f} {('نعم' if r['eligible'] else 'لا'):>6} {end_s:>13}{star}")
champ_st = _st.get(champ["name"], {})
champ_reason = reason_ar.get(champ_st.get("stop_reason") or "natural", "اكتملت")
manifest_note = ""
mf = SEEDVC / "runs" / champ["name"] / "model_manifest.json"
if mf.exists():
    try:
        _m = json.loads(mf.read_text(encoding="utf-8"))
        manifest_note = (f"\nملاحظة: نموذج البطل صُنع من نقطة تفتيش الخطوة "
                         f"{_m.get('effective_steps', '?')} ({_m.get('reason', '?')}) — هذا سليم ومقصود.")
    except Exception:
        pass
print(f"""
لماذا فاز؟ أعلى تحسن في تشابه المتحدث العراقي ({champ['speaker_sim_delta']:+.4f})
مع بقاء حفظ المحتوى داخل حد التسامح ({CONTENT_TOL}).
كيف انتهى تدريبها؟ {champ_reason}.{manifest_note}
البقية محفوظة كاملة في runs/ — يمكن الاستئناف فوق نقطتها لاحقاً بتشغيل جديد.
تقارير كل التجارب تُصدَّر مع حزمة البطل (الخلية 22) للمقارنة المستقبلية.""")

# ---- استماع مقارن لزوج البطل: الأساس مقابل البطل ----
ab_dir = SEEDVC / "runs" / champ["name"] / "ab_eval"
print("استمع (نفس المصدر والمرجع العراقي):")
for tag, label in (("base_pair00.wav", "ناتج النموذج الأساس"),
                   ("ft_pair00.wav", "ناتج البطل المضبوط")):
    f = ab_dir / tag
    if f.exists():
        print(f"— {label}:")
        display(Audio(filename=str(f)))
print("\n[التالي] بنشمارك البطل اللحظي (الخلية 21) — ثم التصدير (22).")


In [ ]:
# ============================================================ #
#  الخلية ٢١ — بنشمارك الاستنساخ اللحظي — على البطل الفائز     #
#                                                              #
#  يقيس على كرتك الفعلي، بالنموذج المضبوط:                      #
#   أ) RTF على مستوى المقطع عند 10/16/30 خطوة تدفق             #
#   ب) **البث الإنتاجي الحقيقي**: بروفايل balanced               #
#      (كتل 180ms + crossfade 40ms + سياق 20ms + 10 خطوات)     #
#      بنفس منطج محرك رافِدَين الإنتاجي — يعطي p50/p95          #
#      وزمن الوصول الشامل المقدَّر                              #
# ============================================================ #
import json
from pathlib import Path

BENCH = ROOT / "bench_stream.py"
BENCH.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Latency benchmark for the fine-tuned Iraqi model.

Part A: utterance-level RTF at diffusion steps {10, 16, 30}.
Part B: production streaming simulation with the `balanced` profile
        (block 180ms / crossfade 40ms / right ctx 20ms / 10 steps) using
        StreamBenchSession - a faithful port of rafidain's production
        engine (which ports seed-vc's real-time-gui.py).

RTF < 1.0 means faster than real time. The published measurement for this
model family on the balanced profile is ~430 ms end-to-end on a decent GPU.
"""
import os, sys, json, time, argparse, random
os.environ.setdefault("HF_HUB_CACHE", "./checkpoints/hf_cache")
sys.path.insert(0, os.environ.get("NB_ROOT", "."))

import numpy as np
import torch
import torchaudio
from rafidain_eval_lib import load_stack, convert_utterance, load_16k, StreamBenchSession

AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus")


def list_audio(d):
    out = []
    for root, _, files in os.walk(d):
        for f in sorted(files):
            if f.lower().endswith(AUDIO_EXTS):
                out.append(os.path.join(root, f))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ft-ckpt", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--test-dir", required=True)
    ap.add_argument("--fallback-dir", default=None)
    ap.add_argument("--out-dir", required=True)
    args = ap.parse_args()

    torch.manual_seed(7)
    random.seed(7)
    np.random.seed(7)

    clips = list_audio(args.test_dir)
    if len(clips) < 2 and args.fallback_dir:
        clips = clips + list_audio(args.fallback_dir)
    assert len(clips) >= 2, "need at least 2 clips (source + reference)"

    # جرب المقاطع بالترتيب حتى ينجح تحميل واحد (مقطع تالف لا يكسر البنشمارك)
    import librosa as _lib
    def _try_load(path, sr=16000, max_sec=None):
        try:
            y = _lib.load(path, sr=sr)[0]
            if max_sec:
                y = y[: int(sr * max_sec)]
            return y if y.size > sr else None   # تجاهل الأقصر من ثانية
        except Exception:
            return None

    src = ref = None
    for c in clips:
        if src is None and _try_load(c) is not None:
            src = c
        elif ref is None and _try_load(c) is not None:
            ref = c
        if src is not None and ref is not None:
            break
    assert src is not None and ref is not None, "no loadable clips found"
    print(f"[bench] source: {os.path.basename(src)}")
    print(f"[bench] reference: {os.path.basename(ref)}")

    print("\n[load] fine-tuned stack ...")
    t0 = time.time()
    stack = load_stack(args.ft_ckpt, args.config, fp16=True)
    model, semantic_fn, f0_fn, vocoder_fn, campplus_model, to_mel, mel_fn_args = stack
    sr = int(mel_fn_args["sampling_rate"])
    print(f"[load] ready in {time.time()-t0:.0f}s")

    os.makedirs(args.out_dir, exist_ok=True)
    report = {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
              "sr": sr, "utterance_rtf": {}, "streaming": {}}

    # ---- Part A: utterance RTF ----
    print("\n[A] utterance-level RTF (convert ~10s clip, model already loaded):")
    for steps in (10, 16, 30):
        # warmup once for kernel autotuning
        convert_utterance(model, semantic_fn, vocoder_fn, campplus_model, to_mel,
                          src, ref, sr, diffusion_steps=steps, fp16=True, max_src_sec=10)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t0 = time.perf_counter()
        wav, out_sr = convert_utterance(model, semantic_fn, vocoder_fn, campplus_model,
                                        to_mel, src, ref, sr, diffusion_steps=steps,
                                        fp16=True, max_src_sec=10)
        dur = wav.size(-1) / out_sr
        wall = time.perf_counter() - t0
        rtf = wall / dur
        report["utterance_rtf"][str(steps)] = {"wall_s": round(wall, 3),
                                               "audio_s": round(dur, 2),
                                               "rtf": round(rtf, 4)}
        print(f"    steps={steps:>2}: {wall:.2f}s wall for {dur:.1f}s audio  ->  RTF {rtf:.3f}"
              + ("  (أسرع من الزمن الحقيقي)" if rtf < 1 else "  (أبطأ من الزمن الحقيقي!)"))

    # ---- Part B: streaming with production profile ----
    profiles = {
        "balanced": dict(block_ms=180, crossfade_ms=40, context_left_ms=2500,
                         context_right_ms=20, diffusion_steps=10,
                         inference_cfg_rate=0.7, max_prompt_s=3.0),
        "ultra_low": dict(block_ms=120, crossfade_ms=40, context_left_ms=2000,
                          context_right_ms=20, diffusion_steps=4,
                          inference_cfg_rate=0.0, max_prompt_s=3.0),
    }
    src16 = load_16k(src, 8.0)   # 8 ثوانٍ من الكلام العراقي الحقيقي
    print("\n[B] streaming simulation (production engine math):")
    for name, prof in profiles.items():
        sess = StreamBenchSession(stack, prof, ref)
        block = sess.block_frame_16k
        times = []
        emitted_all = []
        pos = 0
        n_blocks = 0
        while pos + block <= src16.size:
            pcm = src16[pos: pos + block]
            t0 = time.perf_counter()
            out = sess.convert_block(pcm)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            emitted_all.append(out)
            if n_blocks >= 3:          # skip first blocks (CUDA warmup)
                times.append(dt)
            n_blocks += 1
            pos += block
        emitted_all.append(sess.finish())
        if not times:
            print(f"    [{name}] المصدر أقصر من 3 كتل — تخطّي")
            continue
        if name == "balanced" and emitted_all:
            # احتفظ بصوت البث للاستماع — دليل ملموس أن مسار الإنتاج يعمل بالنقطة المضبوطة
            import numpy as _np
            full = _np.concatenate([e for e in emitted_all if e.size])
            torchaudio.save(os.path.join(args.out_dir, "stream_balanced_out.wav"),
                            torch.from_numpy(full).reshape(1, -1).float(), sess.sr)
            print(f"    [{name}] الصوت المتدفق حُفظ: stream_balanced_out.wav "
                  f"({full.size/sess.sr:.1f}s)")
        times_np = np.array(times)
        p50 = float(np.percentile(times_np, 50)) * 1000
        p95 = float(np.percentile(times_np, 95)) * 1000
        mean = float(times_np.mean()) * 1000
        rtf = float(times_np.mean()) / (prof["block_ms"] / 1000)
        algo = prof["block_ms"] + prof["crossfade_ms"] + prof["context_right_ms"]
        e2e = algo + mean
        report["streaming"][name] = {
            "block_ms": prof["block_ms"], "diffusion_steps": prof["diffusion_steps"],
            "n_blocks_measured": int(times_np.size),
            "block_time_p50_ms": round(p50, 1), "block_time_p95_ms": round(p95, 1),
            "block_time_mean_ms": round(mean, 1),
            "algorithmic_delay_ms": algo,
            "estimated_e2e_ms": round(e2e, 1),
            "rtf": round(rtf, 3),
        }
        print(f"    [{name}] كتلة {prof['block_ms']}ms × {prof['diffusion_steps']} خطوة:")
        print(f"        زمن معالجة الكتلة: p50 {p50:.0f}ms | p95 {p95:.0f}ms | RTF {rtf:.2f}")
        print(f"        زمن الوصول الشامل المقدَّر (كتلة+crossfade+سياق+استدلال): {e2e:.0f}ms"
              + ("  ← داخل ميزانية المكالمات" if e2e < 500 else "  ← فوق الميزانية"))

    out_json = os.path.join(args.out_dir, "bench_stream.json")
    with open(out_json, "w", encoding="utf-8") as fh:
        json.dump(report, fh, ensure_ascii=False, indent=2)
    print(f"\n[bench] report: {out_json}")


if __name__ == "__main__":
    main()
''', encoding="utf-8")
print(f"[سكربت البنشمارك] كُتب: {BENCH}")

# ---------- تشغيل البنشمارك على البطل الفائز ----------
import json
from pathlib import Path

champ = json.loads((ROOT / "champion.json").read_text(encoding="utf-8"))["name"]
FT = SEEDVC / "runs" / champ / "ft_model.pth"
assert FT.exists(), f"لا ft_model.pth للبطل {champ} — راجع خلايا التدريب"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
BENCH_DIR = SEEDVC / "runs" / champ / "bench"
print(f"بنشمارك البطل: {champ}\n")

env = dict(ENV)
env["NB_ROOT"] = str(ROOT)
sh([VENV_PY, BENCH,
    "--ft-ckpt", FT,
    "--config", CFG_TINY,
    "--test-dir", TEST_DIR,
    "--fallback-dir", DATA_DIR,
    "--out-dir", BENCH_DIR],
   cwd=SEEDVC, env=env)

rep = json.loads((BENCH_DIR / "bench_stream.json").read_text(encoding="utf-8"))
b = rep["streaming"].get("balanced", {})
if b:
    print(f"""
خلاصة «الاستنساخ اللحظي» للبطل على {rep['gpu']}:
  • زمن معالجة كتلة البث (p95): {b['block_time_p95_ms']:.0f}ms — أي أن الصوت يخرج أسرع من دخوله بمعامل {b['rtf']:.2f}x
  • زمن الوصول الشامل المقدَّر: {b['estimated_e2e_ms']:.0f}ms (المرجع المنشور للنموذج نفسه: ~430ms)
  • RTF على مستوى المقطع (10 خطوات): {rep['utterance_rtf'].get('10', {}).get('rtf', '?')}
هذه أرقام جهازك الفعلي بنموذج البطل — تُرفع تلقائياً مع الحزمة (الخلية 22).""")

# استمع لناتج البث اللحظي الحقيقي (بروفايل balanced — نفس مسار الإنتاج)
from IPython.display import Audio, display
streamed = BENCH_DIR / "stream_balanced_out.wav"
if streamed.exists():
    print("\nاستمع لناتج البث اللحظي (كتل 180ms متلاحقة عبر محرك الإنتاج — بالبطل):")
    display(Audio(filename=str(streamed)))


In [ ]:
# ============================================================ #
#  الخلية ٢٢ — تصدير حزمة البطل + تقارير كل التجارب            #
#  أوزان البطل + إعداده + تقاريره + مقارنة التجارب كاملة       #
#  في مجلد ثابت على القرص الدائم: exports/<اسم-التشغيل>/      #
# ============================================================ #
import shutil, json
from pathlib import Path

champ = json.loads((ROOT / "champion.json").read_text(encoding="utf-8"))["name"]
CHAMP_DIR = SEEDVC / "runs" / champ
assert (CHAMP_DIR / "ft_model.pth").exists(), f"البطل {champ} بلا ft_model.pth — راجع خلايا التدريب"
EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
print(f"تصدير البطل: {champ}  →  {EXPORT_DIR}\n")

sh([VENV_PY, PIPE, "export",
    "--run-dir", CHAMP_DIR,
    "--config", CFG_TINY,
    "--data-dir", DATA_DIR,
    "--out", EXPORT_DIR])

# ---- توثيق إضافي يُنسخ مع الحزمة (قابلية إعادة الإنتاج جزء من الجودة) ----
extras = [
    CHAMP_DIR / "train_console.log",             # سجل تدريب البطل
    ROOT / "loss_curves_all.png",                # منحنيات كل التجارب مجتمعة
    ROOT / "champion.json",                      # قرار البطل ومقاييسه وكل التجارب
    ROOT / "trials_state.json",                  # حالات التنفيذ الآلي كاملة (نقاط/إيقاف مبكر/ميزانية)
    ROOT / "scheduler.log",                      # سجل المشغّل الآلي — سطر لكل قرار
    ROOT / "nb_config.json",                     # مقابض التشغيل
    ROOT / "repro.json",                         # بصمات git
    DATA_DIR / "_stats.json",
    ROOT / "data" / "duration_hist.png",         # مدرّج البيانات
]
for extra in extras:
    if extra.exists():
        shutil.copy2(extra, EXPORT_DIR / extra.name)
        print(f"  + {extra.name}")

# ---- تقارير كل التجارب (المقارنة الكاملة — أهم إضافة في نسخة التجارب) ----
trials_dir = EXPORT_DIR / "trials_reports"
trials_dir.mkdir(exist_ok=True)
for t in json.loads((ROOT / "trials.json").read_text(encoding="utf-8")):
    src_j = SEEDVC / "runs" / t["name"] / "ab_eval" / "ab_eval.json"
    if src_j.exists():
        d = trials_dir / t["name"]
        d.mkdir(exist_ok=True)
        shutil.copy2(src_j, d / "ab_eval.json")
        print(f"  + trials_reports/{t['name']}/ab_eval.json")

# ---- تقارير البطل نفسها (A/B + البنشمارك) داخل الحزمة ----
for sub, dst_name in ((CHAMP_DIR / "ab_eval", "ab_eval"),
                      (CHAMP_DIR / "bench", "bench_stream")):
    if sub.exists():
        (EXPORT_DIR / dst_name).mkdir(exist_ok=True)
        for f in sub.iterdir():
            if f.is_file():
                shutil.copy2(f, EXPORT_DIR / dst_name / f.name)
        print(f"  + {dst_name}/ ({len(list((EXPORT_DIR / dst_name).iterdir()))} ملفات)")

# ---- دمج كل شيء في training_report.json واحد غني ----
report_path = EXPORT_DIR / "training_report.json"
report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}
if (ROOT / "repro.json").exists():
    report["reproducibility"] = json.loads((ROOT / "repro.json").read_text(encoding="utf-8"))
if (ROOT / "champion.json").exists():
    report["champion"] = json.loads((ROOT / "champion.json").read_text(encoding="utf-8"))
if (ROOT / "trials_state.json").exists():
    _ts = json.loads((ROOT / "trials_state.json").read_text(encoding="utf-8"))
    _meta = {k: _ts[k] for k in ("active_secs", "budget_flagged", "budget_info",
                                  "all_settled", "t_first_launch") if k in _ts}
    _per_trial = {k: {kk: _ts[k].get(kk) for kk in ("status", "stop_reason", "retries",
                     "nan_retries", "done_steps", "started_at", "finished_at")}
                  for k in _ts if k not in ("active_secs", "prev_loop", "budget_flagged",
                                            "budget_info", "all_settled", "t_first_launch")}
    report["autopilot"] = {"meta": _meta, "trials": _per_trial,
                           "early_stop": {k: CFG[k] for k in CFG
                                          if k.startswith("ES_") or k in
                                          ("EARLY_STOP", "NAN_MAX_RETRIES", "BUDGET_USD",
                                           "BUDGET_HOURS", "BUDGET_HARD_STOP")}}
ab_json = EXPORT_DIR / "ab_eval" / "ab_eval.json"
if ab_json.exists():
    report["ab_evaluation"] = json.loads(ab_json.read_text(encoding="utf-8"))["summary"]
    report["ab_evaluation"]["note"] = ("CAMPPlus speaker similarity + XLSR content "
                                       "preservation, held-out Iraqi pairs, identical seeds")
bench_json = EXPORT_DIR / "bench_stream" / "bench_stream.json"
if bench_json.exists():
    report["latency_benchmark"] = json.loads(bench_json.read_text(encoding="utf-8"))
report["training_config"] = {k: CFG[k] for k in CFG if not k.startswith("_") and k != "HF_TOKEN"}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("  + training_report.json (البطل + مقارنة التجارب + البنشمارك والبصمات)")

total = 0
print(f"\nحزمة التصدير: {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        sz = p.stat().st_size
        total += sz
        rel = p.relative_to(EXPORT_DIR)
        print(f"  {str(rel):<52} {sz/1e6:9.2f} MB")
print(f"  {'الإجمالي':<52} {total/1e6:9.2f} MB")


<div dir="rtl">

## الحفظ الدائم — ثلاث طبقات (حتى تنزّلها بعد سنة)

| الطبقة | أين تبقى | كم تدوم | كيف تنزّلها لاحقاً |
|---|---|---|---|
| 1. القرص الشبكي RunPod | `/workspace/iraqi_seedvc/exports/<run>/` | ما دام الـ Volume موجوداً (لا تنسَ ألا تحذفه) | scp من لوحة RunPod ← Connect ← SSH، أو متصفح ملفات Jupyter |
| 2. HuggingFace Hub | مستودع خاص في **حسابك أنت** | **للأبد** ومجاني | `hf download <حسابك>/rafidain-seedvc-iraqi --local-dir ./model` |
| 3. أرشيف tar.gz | ملف واحد على القرص الشبكي | كالطبقة 1 | تنزيل مباشر من متصفح الملفات |

والمحتوى نفسه قابل للرفع لأي سحابة عبر rclone (الخلية 25) — طبقة رابعة اختيارية.

**توصية عملية:** لا تعتمد على طبقة واحدة. الحد الأدنى الموصى به: الطبقة 1 + الطبقة 2. الرفع إلى HuggingFace يستغرق دقائق ويجعل النموذج محصناً ضد أي شيء يحدث لحسابك في RunPod.

</div>


In [ ]:
# ============================================================ #
#  الخلية ٢٣ — أرشيف tar.gz واحد جاهز للتنزيل                  #
# ============================================================ #
import shutil
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (22) أولاً"

base = EXPORTS / f"rafidain_seedvc_{CFG['RUN_NAME']}"
archive = shutil.make_archive(str(base), "gztar", root_dir=str(EXPORTS), base_dir=EXPORT_DIR.name)
size_mb = Path(archive).stat().st_size / 1e6
print(f"الأرشيف جاهز: {archive}  ({size_mb:.1f} MB)")
print(f'''
لتنزيله الآن:
  • من Jupyter: لوحة الملفات (يساراً) → مجلد exports → كليك يمين على الملف → Download
  • أو من جهازك عبر SSH (تجد الأمر في لوحة RunPod → Connect → SSH over TLS):
      scp -P <المنفذ> root@<العنوان>:{archive} .
''')


In [ ]:
# ============================================================ #
#  الخلية ٢٤ — الرفع الدائم إلى HuggingFace Hub                #
#  النموذج يُخزَّن في حسابك أنت، مجاناً، بلا علاقة بـ RunPod     #
#  (أعد تشغيلها في أي وقت مستقبلاً — ولو بعد شهور)              #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
HF_SCRIPT = ROOT / "hf_upload.py"
HF_SCRIPT.write_text(r'''
import os, time
from huggingface_hub import HfApi

token = os.environ["HF_TOKEN"]
export_dir = os.environ["NB_EXPORT_DIR"]
api = HfApi(token=token)
user = api.whoami()["name"]
repo_id = os.environ.get("NB_REPO_ID", "").strip() or f"{user}/rafidain-seedvc-iraqi"
private = os.environ.get("NB_PRIVATE", "1") == "1"
api.create_repo(repo_id, private=private, exist_ok=True)
print(f"[h[رفع] {export_dir} إلى {repo_id} (خاص = {private}) ...")
api.upload_folder(
    folder_path=export_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message=f"Rafidain Iraqi fine-tune ({time.strftime('%Y-%m-%d %H:%M')})",
)
print("[جاهز] https://huggingface.co/" + repo_id)
''', encoding="utf-8")

if not CFG["HF_TOKEN"]:
    print("لا يوجد رمز HuggingFace بعد — لا مشكلة: كل أصولك محفوظة على القرص الشبكي،")
    print("ويمكنك تنفيذ الرفع الدائم في أي وقت لاحق (الخلية تعمل مستقلة):\n")
    print("  1) أنشئ حساباً مجانياً في huggingface.co إن لم يكن لديك")
    print("  2) من hf.co/settings/tokens أنشئ New Token بصلاحية Write وانسخه")
    print('  3) في الخلية 1 اضبط:  CFG["HF_TOKEN"] = "hf_ضع-الرمز-هنا"  ثم أعد تشغيلها')
    print("  4) أعد تشغيل هذه الخلية — سترفع الحزمة كاملة إلى مستودع خاص في حسابك")
else:
    assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (22) أولاً"
    env = dict(ENV)
    env["HF_TOKEN"] = str(CFG["HF_TOKEN"])
    env["NB_EXPORT_DIR"] = str(EXPORT_DIR)
    env["NB_REPO_ID"] = str(CFG.get("HF_REPO_ID", ""))
    env["NB_PRIVATE"] = "1" if CFG["HF_PRIVATE"] else "0"
    sh([VENV_PY, HF_SCRIPT], env=env)
    print('''
[تم] النموذج الآن في حسابك على HuggingFace — بشكل دائم.
لتنزيله في أي وقت مستقبلاً من أي جهاز (بلا RunPod):
    pip install -U "huggingface_hub[cli]"
    hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
أو من بايثون:
    from huggingface_hub import snapshot_download
    snapshot_download("<اسم-حسابك>/rafidain-seedvc-iraqi", local_dir="./mymodel")
أو مباشرة من صفحة المستودع في المتصفح.''')


In [ ]:
# ============================================================ #
#  الخلية ٢٥ — (اختيارية) طبقة رابعة: rclone سحابي             #
#  Google Drive / S3 / Dropbox / OneDrive ...                  #
#  خلية مرجعية — نفّذ الأوامر يدوياً وقت الحاجة                 #
# ============================================================ #

# 1) التثبيت (مرة واحدة على الحاوية):
#      apt-get install -y rclone          # أو أحدث إصدار من rclone.org/install
#
# 2) الإعداد التفاعلي (مرة واحدة — يربط حسابك السحابي):
#      rclone config                      # أنشئ remote باسم مثلاً: mydrive
#      (ملف الإعداد يبقى في /root/.config/rclone/rclone.conf — انسخه إلى
#       /workspace/iraqi_seedvc/ ليبقى مع القرص الدائم بين الحاويات)
#
# 3) رفع حزمة النموذج كاملة (أوزان + تقارير تقييم + بنشمارك):
#      rclone copy /workspace/iraqi_seedvc/exports/{run} mydrive:SeedVC_Iraqi/ -P
#
# 4) تنزيله لاحقاً من أي جهاز آخر فيه rclone مضبوط على الحساب نفسه:
#      rclone copy mydrive:SeedVC_Iraqi ./mymodel -P

print("خلية مرجعية — الأوامر جاهزة في التعليقات أعلاه.")
print("الطبقة الدائمة الأساسية تبقى HuggingFace Hub (الخلية 24) — الأسهل والأضمن.")


In [ ]:
# ============================================================ #
#  الخلية ٢٦ — (اختيارية) تحرير مساحة القرص                    #
#  أ) حذف المصادر الخام القابلة لإعادة الجلب فقط:              #
#     raw/yodas_webm + مخابئ parquet — وتبقى بيانات التدريب    #
#     المجهزة وكل النقاط والحزم كما هي                         #
#  ب) حذف نقاط التفتيش الثقيلة للتجارب غير الفائزة بعد         #
#     تتويج البطل (تبقى ft_model.pth وكل تقارير كل التجارب)    #
#  ملاحظة: علامات .done تبقى، فلن يُعاد التنزيل تلقائياً؛       #
#  لإعادة الجلب مستقبلاً احذف ملف .done المناسب من raw/.done/  #
# ============================================================ #
import shutil, json
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")

def dir_gb(p: Path) -> float:
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

targets = [ROOT / "raw" / "yodas_webm", ROOT / "raw" / "_parquet", ROOT / "raw" / "_safaa_src"]
print(f"المساحة الحرة الآن: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
for t in targets:
    s = dir_gb(t)
    print(f"  {t.name:<12} {s:6.2f} GB" + ("   (قابل لإعادة الجلب بالخليتين 7 و8)" if s > 0 else "   (فارغ)"))

if CFG["CLEAN_RAW_AFTER"]:
    for t in targets:
        if t.exists():
            shutil.rmtree(t)
            print(f"  حُذف: {t.name}")
    if UV:
        sh([UV, "cache", "clean"], check=False)
    else:
        sh([str(VENV_PY), "-m", "pip", "cache", "purge"], check=False)
    print(f"المساحة الحرة بعد التنظيف: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
    print("بيانات التدريب الجاهزة في data/iraqi_finetune — لا حاجة للمصادر الخام إلا لإعادة التجهيز.")
else:
    print('\n(أ) لتفعيل حذف المصادر الخام: اجعل  CFG["CLEAN_RAW_AFTER"] = True  في الخلية 1 ثم أعد هذه الخلية.')

# ---- (ب) نقاط التفتيش الثقيلة للتجارب غير الفائزة ----
print()
champ_name = None
if (ROOT / "champion.json").exists():
    try:
        champ_name = json.loads((ROOT / "champion.json").read_text(encoding="utf-8"))["name"]
    except Exception:
        pass
if champ_name is None:
    print("(ب) تنظيف نقاط التفتيش: يتطلب اختيار البطل أولاً (الخلية 20) — لاحقاً.")
else:
    print(f"(ب) البطل: {champ_name} — نقاطه تبقى كاملة (للاستئناف فوقها مستقبلاً).")
    freed = 0.0
    removed_n = 0
    for t in json.loads((ROOT / "trials.json").read_text(encoding="utf-8")):
        name = t["name"]
        if name == champ_name:
            continue
        tdir = SEEDVC / "runs" / name
        ckpts = list(tdir.glob("DiT_epoch_*_step_*.pth")) if tdir.exists() else []
        if not ckpts:
            continue
        sz = sum(c.stat().st_size for c in ckpts) / 1e9
        keep = (tdir / "ft_model.pth").exists()
        if CFG["CLEAN_CKPTS_AFTER"] and keep:
            for c in ckpts:
                c.unlink()
            freed += sz
            removed_n += len(ckpts)
            print(f"  • {name}: حُذفت {len(ckpts)} نقطة تفتيش ({sz:.2f} GB) — "
                  f"بقي ft_model.pth وكل التقارير")
        elif CFG["CLEAN_CKPTS_AFTER"] and not keep:
            print(f"  • {name}: لا ft_model.pth (فاشلة) — أبقينا نقاطها لاحتمال استئنافها يدوياً")
        else:
            print(f"  • {name}: {len(ckpts)} نقطة تفتيش ({sz:.2f} GB) — تبقى (فعّل CLEAN_CKPTS_AFTER لحذفها)")
    if freed > 0:
        print(f"  المجموع المحرر: {freed:.2f} GB — المساحة الحرة الآن: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
    else:
        print('  لتفعيله: اجعل  CFG["CLEAN_CKPTS_AFTER"] = True  في الخلية 1 ثم أعد هذه الخلية.')


In [ ]:
# ============================================================ #
#  الخلية ٢٧ — النشر: كيف تستخدم النموذج المضبوط؟              #
#  (خلية إرشادية تطبع الأوامر الجاهزة بمساراتك الفعلية)        #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
W = EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth"
C = EXPORT_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

if not W.exists():
    print("شغّل خلية التصدير (22) أولاً حتى تتوفر الحزمة.")
else:
    # اسم البطل الفائز (من خلية الترجيح 20)
    import json
    if (ROOT / "champion.json").exists():
        _ch = json.loads((ROOT / "champion.json").read_text(encoding="utf-8"))
        print(f"بطل هذا التشغيل: {_ch['name']}  ({_ch['decided_by']})")
        print(f"  تحسن تشابه المتحدث: {_ch['metrics']['speaker_sim_delta']:+.4f} "
              f"| حفظ المحتوى: {_ch['metrics']['ft_content_sim_mean']:.4f}")
        print("  (تقارير بقية التجارب للمقارنة: trials_reports/ داخل الحزمة)\n")

    # لوحة نتائج التقييم إن وُجدت (تذكير بأرقام هذا التشغيل)
    ab = EXPORT_DIR / "ab_eval" / "ab_eval.json"
    bench = EXPORT_DIR / "bench_stream" / "bench_stream.json"
    if ab.exists():
        s = json.loads(ab.read_text(encoding="utf-8"))["summary"]
        print(f"أرقام هذا التشغيل: تشابه متحدث {s['base_speaker_sim_mean']:.4f} ← {s['ft_speaker_sim_mean']:.4f} "
              f"(فرق {s['speaker_sim_delta']:+.4f}) | حفظ محتوى {s['ft_content_sim_mean']:.4f}")
    if bench.exists():
        b = json.loads(bench.read_text(encoding="utf-8"))["streaming"].get("balanced", {})
        if b:
            print(f"                   زمن وصول شامل (balanced): {b['estimated_e2e_ms']:.0f}ms | RTF {b['rtf']:.2f}")
    print(f'''
==========================================================
 نشر النموذج المضبوط — ثلاثة مسارات جاهزة
==========================================================

(أ) استدلال فوري في هذه الحاوية (البيئة جاهزة الآن):
    cd {SEEDVC}
    {VENV_PY} inference.py \\
        --checkpoint {W} \\
        --config     {C} \\
        --source <صوت-المتكلم-الأصلي.wav> \\
        --target  <صوت-مرجعي-عراقي.wav> \\
        --output  ./out

(ب) واجهة ويب Gradio للتجربة السريعة (تحتاج تثبيت gradio أولاً):
    uv pip install --python {VENV_PY} gradio   # أو: {VENV_PY} -m pip install gradio
    cd {SEEDVC}
    {VENV_PY} app_vc.py --checkpoint {W} --config {C}
    ثم افتح المنفذ 7860 (فعّل بروكسي HTTP من إعدادات الحاوية في لوحة RunPod)

(ج) خدمة رافِدَين الكاملة (REST + WebSocket لبث لحظي في المكالمات):
    المستودع منسوخ هنا: {RAFIDAIN}
    انسخ ما يلي في ملف .env الخاص بالخدمة (انظر .env.example داخل المستودع):

      RAFIDAIN__ENGINE__LOCAL_CHECKPOINT={W}
      RAFIDAIN__ENGINE__LOCAL_CONFIG={C}
      RAFIDAIN__ENGINE__DEVICE=cuda
      RAFIDAIN__API_KEYS=["<مفتاح-API-عشوائي-خاص-بك>"]

    التفاصيل الكاملة (Docker/GPU، الأمان، TLS، القياس):
      {RAFIDAIN}/DEPLOYMENT.md   و   {RAFIDAIN}/WEBUI.md

تذكير أخلاقي: استنساخ صوت شخص حقيقي يتطلب موافقته الصريحة الموثقة،
وأفصح للطرف الآخر عند ما يوجبه القانون — راجع قسم الأمان في README المستودع.
''')


<div dir="rtl">

## استكشاف الأخطاء — النسخة الآلية (Autopilot)

| المشكلة | الحل |
|---|---|
| `no kernel image is available for execution on the device` | بطاقة Blackwell مع PyTorch قديم خارج البيئة المعزولة. الحل: أعد الخلية 4 (تثبّت 2.8.0 cu128 وتفحص **كل** بطاقة)، وإن استمر: احذف مجلد `venv` تحت الجذر ثم أعد الخلية 4 من الصفر |
| لوحة المراقبة تقول «لا نبض منذ X دقيقة» | المشغّل مات (نادر — عادة OOM نظام أو خلع حاوية). الحل: أعد تشغيل خلية الإطلاق (15) — يقرأ الحالة المحفوظة ويستأنف كل شيء من آخر نقطة تفتيش |
| تجربة حالتها «موقوفة (إيقاف مبكر)» | هذا **النظام يعمل** وليس خطأ: خسارتها استوت فوفّرت عليك المال، ونموذجها صُنع من آخر نقطة سليمة ويُقيَّم كأي نموذج. لتعطيل السلوك: `EARLY_STOP = False` في الخلية 1، أو `es=False` لتجربة محددة في الخلية 14 |
| تجربة حالتها «موقوفة (بعد NaN)» أو «فاشلة (NaN)» | NaN تكرر أكثر من مرتين رغم الاستئناف من نقاط سليمة — خفّض دفعة تلك التجربة في الخلية 14 ثم أعد إطلاقها (اجعل حالتها `pending` في `trials_state.json` وأعد الخلية 15) |
| تجربة «موقوفة (سقف الميزانية)» وأخرى «متجاوزة» | بلغت حد $15/6 ساعات. للاستمرار: ارفع `BUDGET_USD`/`BUDGET_HOURS` في الخلية 1، ثم اجعل حالة المتجاوزة `pending` في `trials_state.json` وأعد الخلية 15 |
| تجربة واحدة «فاشلة (بعد انهيارين)» | راجع سجلها `runs/<التجربة>/train_console.log` — غالباً OOM: خفّض `batch` لها في الخلية 14، ثم `pending` + خلية 15. إن كان لها نقاط سليمة فستجد نموذجاً مصنوعاً جاهزاً للتقييم أصلاً |
| «فشل استئناف التجربة / انهارت فور الإطلاق» بعد خلع الحاوية | قد تكون آخر نقطة تفتيش كُتبت منتصف الحفظ. المشغّل يفحص سلامة كل نقطة ويحذف التالفة تلقائياً قبل الإطلاق — أعد الخلية 15 فقط |
| `CUDA out of memory` أثناء التدريب | الدفعة التلقائية 6 كافية عادة؛ إن شغّلت شيئاً آخر على البطاقة خفّض دفعة تجاربها في الخلية 14 (4 أو 3) ثم أعد الخلية 15 |
| أريد إعادة تجربة فاشلة بعد إصلاح السبب | في `trials_state.json` اجعل `"status"` لتلك التجربة `"pending"` (أو احذف مدخلها) ثم أعد الخلية 15 — تستأنف من آخر نقطة **بالخطوات المتبقية فقط** |
| أريد إعادة تجربة من الصفر تماماً | احذف مجلدها `runs/<التجربة>/` بالكامل ثم `pending` + خلية 15 |
| توقفت الحاوية كلها (خلع Pod) | لا تفعل شيئاً: أعد تشغيل الحاوية، شغّل الخلايا بالترتيب من 1 — كل شيء يستأنف: المشغّل يعيد الميت، والتجربة تكمل من آخر نقطة سليمة، والزمن الفعلي للميزانية لا يحتسب فترات التوقف |
| كيف أعرف ماذا يحدث الآن بالضبط؟ | الخلية 16 (اللوحة الحية) + ملفا `scheduler.log` و`trials_state.json` تحت الجذر — سجل المشغّل فيه سطر لكل قرار (إطلاق/إيقاف/تصنيع/NaN) |
| استأنفت التجربة بخطوات أقل من المحدد — خطأ؟ | لا — هذا هو السلوك الصحيح: المدرب الرسمي يعيد عدّ الخطوات من صفر عند الاستئناف، فالمشغّل يطلق **المتبقي فقط** لتبقى الجرعة الكلية كما رسمت في الشبكة |
| أريد تعديل التجارب أو إضافة واحدة | عدّل قائمة `TRIALS` في الخلية 14 وأعد تشغيلها ثم الخلية 15 — المشغّل يقرأ الشبكة الجديدة كل دورة: المنجزة تبقى، والجديدة تبدأ، والمعدّلة تحتاج حذف مجلدها في `runs/` |
| بطاقة واحدة فقط — هل الدفتر مناسب؟ | نعم — الوضع التسلسلي الآلي وضع رسمي كامل: توزيع تلقائي، وكل الضمانات. بطاقة ثانية تضيف الموازاة فقط |
| قائمة CUDA في RunPod تعرض 13.2 فقط | لا تأثير — بيئة الدفتر معزولة وتحمل مكتبات CUDA 12.8 داخل حزم pip، والدرايفر أحدث من كليهما. اختر وامضِ |
| فشل تنزيل YODAS لبعض المصادر | أعد الخلية 8 — تعيد محاولة الفاشل فقط. عدد قليل من الفشل لا يؤثر (الكتالوج فيه 2360 مصدراً) |
| بطء تنزيل HuggingFace | اضبط `CFG["HF_ENDPOINT"] = "https://hf-mirror.com"` في الخلية 1 |
| نفاد مساحة القرص | فعّل `CLEAN_RAW_AFTER` (الخلية 26) للمصادر الخام، أو `CLEAN_CKPTS_AFTER` لنقاط التجارب غير الفائزة بعد تتويج البطل، أو وسّع الـ Volume من لوحة RunPod |
| كل التجارب تقول NEUTRAL | البيانات أو الجرعة غير كافية بعد: زد الخطوات في الشبكة (الخلية 14) وأعد الإطلاق — كل تجربة تستأنف فوق نقطتها |
| النتيجة جيدة بالأرقام لكن الصوت «معدني» | عادة الفوكودر لا الـ DiT — ارفع خطوات التدفق عند الاستدلال (`--diffusion-steps 30`) |
| أردت بطراً غير الذي اختارته الخلية 20 | اضبط `CFG["CHAMPION"] = "<اسم-التجربة>"` في الخلية 1 ثم أعد الخلية 20 — أو عدّل `champion.json` يدوياً |
| أعدت تشغيل النواة وضاعت المتغيرات | طبيعي — شغّل الخلايا بالترتيب من 1: كل خلية تكتشف المنجز وتتخطاه |

## الخلاصة — أين كل شيء؟ (خريطة المقتنيات الدائمة)

| الأصل | المسار الدائم (على القرص الشبكي) |
|---|---|
| حزمة البطل (أوزان + إعداد + تقاريره + مقارنة التجارب) | `exports/<اسم-التشغيل>/` |
| تقارير كل التجارب للمقارنة المستقبلية | `exports/<اسم-التشغيل>/trials_reports/` |
| الأرشيف المضغوط الواحد | `exports/rafidain_seedvc_<اسم-التشغيل>.tar.gz` |
| نسخة HuggingFace (للأبد) | `hf.co/<حسابك>/rafidain-seedvc-iraqi` |
| نقاط وسجلات كل تجربة + قراراتها | `seed-vc/runs/<التجربة>/` |
| توثيق النموذج المصنّع (من أي نقطة ولماذا) | `seed-vc/runs/<التجربة>/model_manifest.json` |
| حالة المشغّل الآلي (نبض/إيقاف/ميزانية) + سجل قراراته | `trials_state.json` + `scheduler.log` |
| منحنيات كل التجارب مجتمعة (بعلامات أفضل نقطة) | `loss_curves_all.png` (وداخل الحزمة) |
| قرار البطل ومقاييس الجميع | `champion.json` (وداخل الحزمة) |
| بيانات التدريب المجهزة + المجموعة الفرعية | `data/iraqi_finetune/` + `data/iraqi_finetune_50pct/` |
| بيانات الاختبار المعزولة | `data/iraqi_test/` |
| كل إعدادات التشغيل + بصمات git | `nb_config.json` + `repro.json` (وداخل الحزمة) |

**آخر خطوة دائماً:** شغّل الخلية 24 (HuggingFace) — دقيقتان تشتريانك راحة بال دائمة.

</div>
